# 1. Environment Configuration and Library Imports
Define the working directory in the synchronized Google Drive folder (`/home/daves/google_drive`) inside WSL, and import dependencies for tabular processing, out-of-core SQL analysis (DuckDB), web geocoding, and interactive map rendering.

In [1]:
import os
import glob
import time
import zipfile
import requests
import duckdb
import numpy as np
import pandas as pd
import folium
from folium.plugins import FastMarkerCluster

# Working directory on Google Drive mounted in WSL
base_dir = "/home/daves/google_drive/Pessoal/Notebooks/Geografia do voto"
os.makedirs(base_dir, exist_ok=True)
os.chdir(base_dir)

# Default Windows downloads path accessible via WSL
downloads_windows = "/mnt/c/Users/daves/Downloads"

print(f"Active working directory: {os.getcwd()}")

Active working directory: /home/daves/google_drive/Pessoal/Notebooks/Geografia do voto


# 2. Consolidation and Cleansing of Voting Locations (All States + Abroad)
Dynamically load all state-level CSV files (`eleitorado_local_votacao_2026_*.csv`), unifying all 26 states, the Federal District, and overseas voting (`ZZ`). Sections repeated across rounds are deduplicated giving priority to the 2nd round record (`NR_TURNO = 2`), since the TSE moved part of the sections to new venues for the runoff. Coordinates (`NR_LATITUDE`, `NR_LONGITUDE`) are converted to numeric floats and join keys to integers.

In [2]:
url_locais = "https://cdn.tse.jus.br/estatistica/sead/odsele/eleitorado_locais_votacao/eleitorado_local_votacao_2026.zip"
zip_locais = os.path.join(base_dir, "eleitorado_local_votacao_2026.zip")
locais_dir = os.path.join(base_dir, "eleitorado_local_votacao_2026")

if not os.path.exists(locais_dir) or len(os.listdir(locais_dir)) == 0:
    os.makedirs(locais_dir, exist_ok=True)
    print("Downloading eleitorado_local_votacao_2026.zip from TSE repository...")
    with requests.get(url_locais, stream=True) as r:
        r.raise_for_status()
        with open(zip_locais, 'wb') as f:
            for chunk in r.iter_content(chunk_size=2 * 1024 * 1024):
                if chunk:
                    f.write(chunk)

    print("Extracting voting location files...")
    with zipfile.ZipFile(zip_locais, 'r') as zip_ref:
        zip_ref.extractall(locais_dir)
    print(f"Extraction completed successfully at: {locais_dir}")
else:
    print(f"Voting locations directory already available at: {locais_dir}")

# Find all CSV files (per-state files including ZZ, or a single national file)
arquivos_locais = sorted(glob.glob(os.path.join(locais_dir, "**", "eleitorado_local_votacao_2026*.csv"), recursive=True))

if not arquivos_locais:
    raise FileNotFoundError(f"No CSV files found in: {locais_dir}")

cols_locais = [
    'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO',
    'NR_LATITUDE', 'NR_LONGITUDE', 'NM_LOCAL_VOTACAO', 'DS_ENDERECO',
    'CD_TIPO_SECAO_AGREGADA', 'NR_SECAO_PRINCIPAL'
]

# Keep the round indicator when the release provides it
cols_header = pd.read_csv(arquivos_locais[0], sep=';', encoding='latin1', nrows=0).columns.tolist()
tem_turno = 'NR_TURNO' in cols_header
if tem_turno:
    cols_locais.append('NR_TURNO')

print(f"Loading and combining {len(arquivos_locais)} voting location files (States + ZZ) | NR_TURNO available: {tem_turno}")

lista_dfs_locais = []
for f in arquivos_locais:
    df_temp = pd.read_csv(f, sep=';', encoding='latin1', usecols=cols_locais, low_memory=False)
    lista_dfs_locais.append(df_temp)

df_geo = pd.concat(lista_dfs_locais, ignore_index=True)

# 1. Deduplicate sections repeated across rounds, keeping the 2nd round venue first
if tem_turno:
    df_geo['NR_TURNO'] = pd.to_numeric(df_geo['NR_TURNO'], errors='coerce')
    df_geo = df_geo.sort_values('NR_TURNO', ascending=False)
df_geo = df_geo.drop_duplicates(subset=['SG_UF', 'CD_MUNICIPIO', 'NR_ZONA', 'NR_SECAO']).copy()

# 2. Standardize decimal commas to dots and cast coordinates to float
df_geo['lat_clean'] = pd.to_numeric(df_geo['NR_LATITUDE'].astype(str).str.replace(',', '.'), errors='coerce')
df_geo['lon_clean'] = pd.to_numeric(df_geo['NR_LONGITUDE'].astype(str).str.replace(',', '.'), errors='coerce')

# 3. Ensure uniform types for join keys and aggregation code
for col in ['CD_MUNICIPIO', 'NR_ZONA', 'NR_SECAO']:
    df_geo[col] = pd.to_numeric(df_geo[col], errors='coerce').fillna(0).astype(int)
df_geo['CD_TIPO_SECAO_AGREGADA'] = pd.to_numeric(df_geo['CD_TIPO_SECAO_AGREGADA'], errors='coerce')
df_geo['SG_UF'] = df_geo['SG_UF'].astype(str).str.strip().str.upper()

print(f"Total unique voting sections consolidated: {len(df_geo):,}")

Voting locations directory already available at: /home/daves/google_drive/Pessoal/Notebooks/Geografia do voto/eleitorado_local_votacao_2026
Loading and combining 29 voting location files (States + ZZ) | NR_TURNO available: True
Total unique voting sections consolidated: 517,179


# 3. Aggregation of Registered Voters by Section via DuckDB
Process the voting location CSVs out-of-core using DuckDB, extracting the registered voters per section (`QT_ELEITOR_SECAO`) for each unique electoral key (`SG_UF`, `CD_MUNICIPIO`, `NR_ZONA`, `NR_SECAO`). Sections repeated across rounds are collapsed with `MAX`, which avoids double counting.

In [3]:
con = duckdb.connect()

lista_sql = "[" + ", ".join(f"'{f}'" for f in arquivos_locais) + "]"
fonte_csv = f"read_csv({lista_sql}, delim=';', encoding='latin-1', header=True, all_varchar=True, union_by_name=True)"

# Identify the voter count column available in this release
colunas_disponiveis = con.execute(f"DESCRIBE SELECT * FROM {fonte_csv}").df()['column_name'].tolist()
colunas_qt = [c for c in colunas_disponiveis if c.startswith('QT_ELEITOR')]
col_qt = 'QT_ELEITOR_SECAO' if 'QT_ELEITOR_SECAO' in colunas_disponiveis else colunas_qt[0]
print(f"Voter count column: {col_qt} | Available QT columns: {colunas_qt}")

query_perfil = f'''
SELECT
    UPPER(TRIM(SG_UF)) AS SG_UF,
    CAST(CD_MUNICIPIO AS INTEGER) AS CD_MUNICIPIO,
    CAST(NR_ZONA AS INTEGER) AS NR_ZONA,
    CAST(NR_SECAO AS INTEGER) AS NR_SECAO,
    MAX(TRY_CAST({col_qt} AS BIGINT)) AS QT_ELEITORES_TOTAL
FROM {fonte_csv}
GROUP BY 1, 2, 3, 4
'''

df_perfil_total = con.execute(query_perfil).df()

for col in ['CD_MUNICIPIO', 'NR_ZONA', 'NR_SECAO']:
    df_perfil_total[col] = df_perfil_total[col].astype(int)

print(f"Total sections with computed voter count: {len(df_perfil_total):,}")
print(f"Total registered voters: {df_perfil_total['QT_ELEITORES_TOTAL'].sum():,}")

Voter count column: QT_ELEITOR_SECAO | Available QT columns: ['QT_ELEITOR_SECAO', 'QT_ELEITOR_ELEICAO_FEDERAL', 'QT_ELEITOR_ELEICAO_ESTADUAL', 'QT_ELEITOR_ELEICAO_MUNICIPAL']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total sections with computed voter count: 517,179
Total registered voters: 158,745,463


# 4. First Merge and Spatial Error Audit
Execute an outer join between section voter counts and georeferenced voting venues. Classify and catalog data quality issues: sentinel coordinates (`-1`), missing values, coordinates outside the Brazilian bounding box, aggregated sections, and overseas voting (`ZZ`).

The aggregated-section rule uses the TSE coding of `CD_TIPO_SECAO_AGREGADA`: `1` = principal section, `2` = aggregated section (its voters vote in the ballot box of `NR_SECAO_PRINCIPAL`).

In [4]:
chaves_join = ['SG_UF', 'CD_MUNICIPIO', 'NR_ZONA', 'NR_SECAO']

rotulo_ok = 'OK: Georeferenced Brazil'
rotulo_perfil_sem_local = 'Error: Section in profile missing from locations catalog'
rotulo_local_sem_perfil = 'Warning: Location has no registered voters in profile'
rotulo_zz = 'Info: Overseas Voting (ZZ) - International coordinates or no GPS'
rotulo_menos1 = 'Error: Coordinate -1 (Unrecorded by TSE)'
rotulo_nan = 'Error: Coordinate Null/NaN'
rotulo_fora_bbox = 'Error: Coordinate outside Brazilian Bounding Box'
rotulo_agregada = 'Warning: Aggregated section (shares physical ballot box)'

# First relational merge (outer)
df_merged = pd.merge(
    df_perfil_total,
    df_geo,
    on=chaves_join,
    how='outer',
    indicator=True
)

# Geographic validation rules (evaluated in order; first match wins)
condicoes = [
    df_merged['_merge'] == 'left_only',
    df_merged['_merge'] == 'right_only',
    (df_merged['SG_UF'] == 'ZZ') & (
        (df_merged['lat_clean'] == -1) |
        (df_merged['lat_clean'].isna()) |
        (~df_merged['lat_clean'].between(-35.0, 6.0))
    ),
    (df_merged['lat_clean'] == -1) | (df_merged['lon_clean'] == -1),
    df_merged['lat_clean'].isna() | df_merged['lon_clean'].isna(),
    (~df_merged['lat_clean'].between(-35.0, 6.0)) | (~df_merged['lon_clean'].between(-75.0, -34.0)),
    df_merged['CD_TIPO_SECAO_AGREGADA'] == 2
]

rotulos = [
    rotulo_perfil_sem_local,
    rotulo_local_sem_perfil,
    rotulo_zz,
    rotulo_menos1,
    rotulo_nan,
    rotulo_fora_bbox,
    rotulo_agregada
]

df_merged['STATUS_GEO'] = np.select(condicoes, rotulos, default=rotulo_ok)

# Display audit summary
tabela_erros = df_merged['STATUS_GEO'].value_counts().reset_index()
tabela_erros.columns = ['Section Diagnosis', 'Count']
print("\n=== GEOGRAPHIC AUDIT SUMMARY (FIRST MERGE) ===")
print(tabela_erros.to_string(index=False))

# Export detailed audit table to CSV
caminho_auditoria = os.path.join(base_dir, "tabela_erros_georreferenciamento.csv")
df_merged[df_merged['STATUS_GEO'] != rotulo_ok].to_csv(
    caminho_auditoria, sep=';', index=False, encoding='utf-8'
)
print(f"\nInconsistency report saved to: {caminho_auditoria}")


=== GEOGRAPHIC AUDIT SUMMARY (FIRST MERGE) ===
                                               Section Diagnosis  Count
                                        OK: Georeferenced Brazil 496015
Info: Overseas Voting (ZZ) - International coordinates or no GPS   2715
                        Error: Coordinate -1 (Unrecorded by TSE)   2146
                Error: Coordinate outside Brazilian Bounding Box     18

Inconsistency report saved to: /home/daves/google_drive/Pessoal/Notebooks/Geografia do voto/tabela_erros_georreferenciamento.csv


# 5. Audit Actions: Second Merge and Web Geocoding
Decisions taken for each diagnosis of the first audit:

| Diagnosis | Action |
|---|---|
| Aggregated section (shares physical ballot box) | Not treated; kept with the TSE coordinate |
| Overseas Voting (ZZ) | Included (global map); all records carry `-1`, so they are geocoded |
| Coordinate `-1` (Unrecorded by TSE) | Geocoded from the CEP (e.g. `69970000`) / municipality and a web search for `NM_LOCAL_VOTACAO` |
| Coordinate outside Brazilian Bounding Box | Included (global map), kept with the TSE coordinate |

The second merge attaches `NR_LOCAL_VOTACAO`, `NR_CEP` and `NM_BAIRRO` from the raw TSE files. Geocoding runs once per physical venue in the cascade below; the method that produced each coordinate is recorded in `GEO_METODO`:

1. `nominatim_venue_name`: Nominatim/OSM search by `NM_LOCAL_VOTACAO` + municipality + UF (accepted only if the result falls in the same UF);
2. `nominatim_address`: Nominatim search by `DS_ENDERECO` + `NM_BAIRRO` + municipality + UF;
3. `cep_centroid`: CEP centroid (AwesomeAPI), validated by UF;
4. `municipality_centroid`: municipality (or foreign city) centroid;
5. `unresolved`: no source returned a valid coordinate.

Sections that keep the TSE coordinate receive `GEO_METODO = tse_original`. Results are appended to `cache_geocodificacao_locais.csv` after each venue, so an interrupted run resumes where it stopped (Nominatim allows at most 1 request per second).

In [5]:
# 1. Second merge: venue attributes needed for geocoding (venue id, CEP, neighborhood)
ordem_turno = "ORDER BY TRY_CAST(NR_TURNO AS INTEGER) DESC" if 'NR_TURNO' in colunas_disponiveis else ""

df_extra = con.execute(f'''
SELECT
    UPPER(TRIM(SG_UF)) AS SG_UF,
    CAST(CD_MUNICIPIO AS INTEGER) AS CD_MUNICIPIO,
    CAST(NR_ZONA AS INTEGER) AS NR_ZONA,
    CAST(NR_SECAO AS INTEGER) AS NR_SECAO,
    CAST(NR_LOCAL_VOTACAO AS INTEGER) AS NR_LOCAL_VOTACAO,
    CASE WHEN TRIM(NR_CEP) LIKE '-%' THEN NULL
         ELSE LPAD(REGEXP_REPLACE(NR_CEP, '[^0-9]', '', 'g'), 8, '0') END AS NR_CEP,
    TRIM(NM_BAIRRO) AS NM_BAIRRO
FROM {fonte_csv}
{ordem_turno}
''').df().drop_duplicates(subset=chaves_join)

df_merged = df_merged.drop(columns=['NR_LOCAL_VOTACAO', 'NR_CEP', 'NM_BAIRRO'], errors='ignore')
df_merged = df_merged.merge(df_extra, on=chaves_join, how='left')
print(f"Second merge completed: {len(df_merged):,} sections | without NR_LOCAL_VOTACAO: {df_merged['NR_LOCAL_VOTACAO'].isna().sum():,}")

# 2. Geocoding queue: one row per physical venue (-1 in Brazil + all ZZ)
chaves_local = ['SG_UF', 'CD_MUNICIPIO', 'NR_ZONA', 'NR_LOCAL_VOTACAO']
status_geocodificar = [rotulo_menos1, rotulo_zz]

df_fila = (
    df_merged[df_merged['STATUS_GEO'].isin(status_geocodificar)]
    .drop_duplicates(subset=chaves_local)
    [chaves_local + ['NM_MUNICIPIO', 'NM_LOCAL_VOTACAO', 'DS_ENDERECO', 'NM_BAIRRO', 'NR_CEP']]
    .fillna('')
    .reset_index(drop=True)
)

# 3. Disk cache (allows resuming an interrupted run)
caminho_cache = os.path.join(base_dir, "cache_geocodificacao_locais.csv")
cols_cache = chaves_local + ['GEO_LAT', 'GEO_LON', 'GEO_METODO', 'GEO_CONSULTA']
if not os.path.exists(caminho_cache):
    pd.DataFrame(columns=cols_cache).to_csv(caminho_cache, sep=';', index=False, encoding='utf-8')

df_cache = pd.read_csv(caminho_cache, sep=';', encoding='utf-8')
chaves_feitas = set(df_cache[chaves_local].astype(str).agg('|'.join, axis=1))
df_fila['CHAVE'] = df_fila[chaves_local].astype(str).agg('|'.join, axis=1)
df_pendente = df_fila[~df_fila['CHAVE'].isin(chaves_feitas)]

print(f"Venues in queue: {len(df_fila):,} | Already cached: {len(df_fila) - len(df_pendente):,} | Pending: {len(df_pendente):,}")

# 4. Cascade geocoding: venue name -> address -> CEP centroid -> municipality centroid
sessao = requests.Session()
sessao.headers.update({'User-Agent': 'geografia-do-voto-research/1.0'})
url_nominatim = "https://nominatim.openstreetmap.org/search"
cache_municipios = {}
inicio = time.time()

for i, row in enumerate(df_pendente.itertuples(index=False), start=1):
    exterior = row.SG_UF == 'ZZ'
    sufixo = row.NM_MUNICIPIO if exterior else f"{row.NM_MUNICIPIO}, {row.SG_UF}, Brasil"

    consultas = [('nominatim_venue_name', f"{row.NM_LOCAL_VOTACAO}, {sufixo}")]
    if not exterior and row.DS_ENDERECO:
        partes_endereco = [p for p in [row.DS_ENDERECO, row.NM_BAIRRO] if p]
        consultas.append(('nominatim_address', ", ".join(partes_endereco) + f", {sufixo}"))

    lat, lon, metodo, consulta_usada = np.nan, np.nan, 'unresolved', ''

    for metodo_i, consulta in consultas:
        params = {'q': consulta, 'format': 'jsonv2', 'limit': 1, 'addressdetails': 1}
        if not exterior:
            params['countrycodes'] = 'br'
        r = sessao.get(url_nominatim, params=params, timeout=30)
        time.sleep(1.1)  # Nominatim usage policy: max 1 request/second
        resultados = r.json() if r.status_code == 200 else []
        if resultados:
            uf_resultado = resultados[0].get('address', {}).get('ISO3166-2-lvl4', '')
            if exterior or uf_resultado == f"BR-{row.SG_UF}":
                lat, lon = float(resultados[0]['lat']), float(resultados[0]['lon'])
                metodo, consulta_usada = metodo_i, consulta
                break

    # CEP centroid (Brazil only; generic CEPs such as 69970000 resolve to the city)
    if metodo == 'unresolved' and not exterior and len(row.NR_CEP) == 8:
        r = sessao.get(f"https://cep.awesomeapi.com.br/json/{row.NR_CEP}", timeout=30)
        time.sleep(0.3)
        dados_cep = r.json() if r.status_code == 200 else {}
        if dados_cep.get('lat') and dados_cep.get('lng') and dados_cep.get('state') == row.SG_UF:
            lat, lon = float(dados_cep['lat']), float(dados_cep['lng'])
            metodo, consulta_usada = 'cep_centroid', row.NR_CEP

    # Municipality / foreign city centroid (last resort)
    if metodo == 'unresolved':
        chave_mun = (row.SG_UF, row.NM_MUNICIPIO)
        if chave_mun not in cache_municipios:
            params = {'q': sufixo, 'format': 'jsonv2', 'limit': 1, 'addressdetails': 1}
            if not exterior:
                params['countrycodes'] = 'br'
            r = sessao.get(url_nominatim, params=params, timeout=30)
            time.sleep(1.1)
            resultados = r.json() if r.status_code == 200 else []
            uf_ok = bool(resultados) and (exterior or resultados[0].get('address', {}).get('ISO3166-2-lvl4', '') == f"BR-{row.SG_UF}")
            cache_municipios[chave_mun] = (float(resultados[0]['lat']), float(resultados[0]['lon'])) if uf_ok else (np.nan, np.nan)
        lat, lon = cache_municipios[chave_mun]
        if not np.isnan(lat):
            metodo, consulta_usada = 'municipality_centroid', sufixo

    pd.DataFrame([{
        'SG_UF': row.SG_UF, 'CD_MUNICIPIO': row.CD_MUNICIPIO, 'NR_ZONA': row.NR_ZONA,
        'NR_LOCAL_VOTACAO': row.NR_LOCAL_VOTACAO, 'GEO_LAT': lat, 'GEO_LON': lon,
        'GEO_METODO': metodo, 'GEO_CONSULTA': consulta_usada
    }]).to_csv(caminho_cache, sep=';', mode='a', header=False, index=False, encoding='utf-8')

    if i % 10 == 0 or i == len(df_pendente):
        minutos = (time.time() - inicio) / 60
        print(f"  {i:,}/{len(df_pendente):,} venues processed | elapsed {minutos:.1f} min | last method: {metodo}")

# 5. Apply geocoded coordinates back to sections and register GEO_METODO
df_cache = pd.read_csv(caminho_cache, sep=';', encoding='utf-8').drop_duplicates(subset=chaves_local, keep='last')
for col in ['CD_MUNICIPIO', 'NR_ZONA', 'NR_LOCAL_VOTACAO']:
    df_cache[col] = df_cache[col].astype(int)
df_cache['SG_UF'] = df_cache['SG_UF'].astype(str)

df_merged = df_merged.drop(columns=['GEO_LAT', 'GEO_LON', 'GEO_METODO', 'GEO_CONSULTA'], errors='ignore')
df_merged['NR_LOCAL_VOTACAO'] = df_merged['NR_LOCAL_VOTACAO'].astype('Int64')
df_cache['NR_LOCAL_VOTACAO'] = df_cache['NR_LOCAL_VOTACAO'].astype('Int64')
df_merged = df_merged.merge(df_cache, on=chaves_local, how='left')

# Keep the original TSE coordinates for traceability before overwriting
df_merged['lat_tse'] = df_merged['lat_clean']
df_merged['lon_tse'] = df_merged['lon_clean']

mask_fila = df_merged['STATUS_GEO'].isin(status_geocodificar)
mask_geocodificado = mask_fila & df_merged['GEO_LAT'].notna()
df_merged.loc[mask_geocodificado, 'lat_clean'] = df_merged.loc[mask_geocodificado, 'GEO_LAT']
df_merged.loc[mask_geocodificado, 'lon_clean'] = df_merged.loc[mask_geocodificado, 'GEO_LON']
df_merged.loc[~mask_fila, 'GEO_METODO'] = 'tse_original'
df_merged.loc[~mask_fila, 'GEO_CONSULTA'] = ''
df_merged['GEO_METODO'] = df_merged['GEO_METODO'].fillna('unresolved')

print(f"\nSections with coordinates replaced by geocoding: {mask_geocodificado.sum():,}")
print(df_merged['GEO_METODO'].value_counts().to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Second merge completed: 517,179 sections | without NR_LOCAL_VOTACAO: 0
Venues in queue: 1,294 | Already cached: 1,294 | Pending: 0

Sections with coordinates replaced by geocoding: 4,860
GEO_METODO
tse_original             512318
municipality_centroid      1987
cep_centroid               1500
nominatim_venue_name       1074
nominatim_address           299
unresolved                    1


## 5.1 Re-audit after Geocoding
Recompute the diagnosis of every section from the corrected coordinates and `GEO_METODO`. The final status (`STATUS_GEO_FINAL`) separates exact TSE coordinates, venue-level geocoding (name or address), approximate centroids (CEP or municipality), coordinates kept outside the Brazilian bounding box, and unresolved records. The cross-tabulation against the first audit (`STATUS_GEO`) documents how each original diagnosis was resolved. The dataset used by the global map (`df_mapa_global`) keeps every section with a valid coordinate.

In [6]:
metodos_venue = ['nominatim_venue_name', 'nominatim_address']
metodos_aproximados = ['cep_centroid', 'municipality_centroid']

coord_valida = (
    df_merged['lat_clean'].notna() & df_merged['lon_clean'].notna()
    & (df_merged['lat_clean'] != -1) & (df_merged['lon_clean'] != -1)
    & df_merged['lat_clean'].between(-90.0, 90.0) & df_merged['lon_clean'].between(-180.0, 180.0)
)
dentro_bbox_br = df_merged['lat_clean'].between(-35.0, 6.0) & df_merged['lon_clean'].between(-75.0, -34.0)
exterior = df_merged['SG_UF'] == 'ZZ'

condicoes_final = [
    df_merged['_merge'] != 'both',
    ~coord_valida,
    df_merged['GEO_METODO'].isin(metodos_venue) & exterior,
    df_merged['GEO_METODO'].isin(metodos_venue),
    df_merged['GEO_METODO'].isin(metodos_aproximados) & exterior,
    df_merged['GEO_METODO'].isin(metodos_aproximados),
    (df_merged['GEO_METODO'] == 'tse_original') & ~dentro_bbox_br,
    (df_merged['GEO_METODO'] == 'tse_original') & (df_merged['STATUS_GEO'] == rotulo_agregada),
    (df_merged['GEO_METODO'] == 'tse_original') & dentro_bbox_br
]

rotulos_final = [
    'Error: Section not matched between voter count and locations',
    'Error: Unresolved coordinate (excluded from map)',
    'OK: Geocoded overseas venue (name)',
    'OK: Geocoded Brazil venue (name/address)',
    'Approx: Overseas city centroid',
    'Approx: Brazil CEP/municipality centroid',
    'Kept: TSE coordinate outside Brazilian Bounding Box',
    'OK: Aggregated section (TSE coordinate)',
    'OK: TSE original coordinate'
]

df_merged['STATUS_GEO_FINAL'] = np.select(condicoes_final, rotulos_final, default='Error: Unclassified')

# Final audit summary (sections and voters)
tabela_reauditoria = (
    df_merged.groupby('STATUS_GEO_FINAL')
    .agg(Sections=('NR_SECAO', 'count'), Voters=('QT_ELEITORES_TOTAL', 'sum'))
    .sort_values('Sections', ascending=False)
    .reset_index()
)
tabela_reauditoria['Pct_Sections'] = (tabela_reauditoria['Sections'] / tabela_reauditoria['Sections'].sum() * 100).round(3)
tabela_reauditoria['Pct_Voters'] = (tabela_reauditoria['Voters'] / tabela_reauditoria['Voters'].sum() * 100).round(3)

print("=== GEOGRAPHIC RE-AUDIT SUMMARY (AFTER GEOCODING) ===")
print(tabela_reauditoria.to_string(index=False))

print("\n=== FIRST AUDIT (rows) x GEO_METODO (columns) ===")
print(pd.crosstab(df_merged['STATUS_GEO'], df_merged['GEO_METODO'], margins=True).to_string())

# Dataset for the global map: every section with a valid coordinate
df_mapa_global = df_merged[coord_valida & (df_merged['_merge'] == 'both')].copy()
print(f"\nSections available for global map: {len(df_mapa_global):,} of {len(df_merged):,} "
      f"({len(df_mapa_global) / len(df_merged) * 100:.3f}%)")
print(f"Registered voters covered: {df_mapa_global['QT_ELEITORES_TOTAL'].sum():,} of {df_merged['QT_ELEITORES_TOTAL'].sum():,}")

# Export re-audit tables
caminho_reauditoria = os.path.join(base_dir, "tabela_reauditoria_georreferenciamento.csv")
df_merged[~df_merged['STATUS_GEO_FINAL'].isin(['OK: TSE original coordinate', 'OK: Aggregated section (TSE coordinate)'])].to_csv(
    caminho_reauditoria, sep=';', index=False, encoding='utf-8'
)

caminho_pendentes = os.path.join(base_dir, "locais_nao_geocodificados.csv")
df_merged[df_merged['STATUS_GEO_FINAL'] == 'Error: Unresolved coordinate (excluded from map)'].drop_duplicates(subset=chaves_local).to_csv(
    caminho_pendentes, sep=';', index=False, encoding='utf-8'
)

print(f"\nRe-audit report saved to: {caminho_reauditoria}")
print(f"Unresolved venues saved to: {caminho_pendentes}")

=== GEOGRAPHIC RE-AUDIT SUMMARY (AFTER GEOCODING) ===
                                   STATUS_GEO_FINAL  Sections    Voters  Pct_Sections  Pct_Voters
                        OK: TSE original coordinate    496015 155266239        95.908      97.808
            OK: Aggregated section (TSE coordinate)     16285   2028512         3.149       1.278
                     Approx: Overseas city centroid      1891    634410         0.366       0.400
           Approx: Brazil CEP/municipality centroid      1596    376383         0.309       0.237
                 OK: Geocoded overseas venue (name)       823    284456         0.159       0.179
           OK: Geocoded Brazil venue (name/address)       550    149977         0.106       0.094
Kept: TSE coordinate outside Brazilian Bounding Box        18      5476         0.003       0.003
   Error: Unresolved coordinate (excluded from map)         1        10         0.000       0.000

=== FIRST AUDIT (rows) x GEO_METODO (columns) ===
GEO_METODO   

## 5.2 Manual Geocoding of Venues Unresolved by the Automatic Cascade
Venues left as `unresolved` after section 5 are geocoded manually from a web search of the official address. Corrections are kept in a table inside this cell (not in the cache), so they are reapplied on every run and are traceable through `GEO_METODO = manual_web_search` and `GEO_CONSULTA` (address and source). After the corrections, `STATUS_GEO_FINAL`, the re-audit summary, the exported tables and `df_mapa_global` are recomputed.

| Venue | Address | Latitude | Longitude | Source |
|---|---|---|---|---|
| Consulado do Brasil em Saint-Georges de l'Oyapock (ZZ, `NR_LOCAL_VOTACAO` 1015) | 15, Rue Jean Cedia, Angle 01 Henri, 97313 Saint-Georges de l'Oyapock, French Guiana | 3.8893706 | -51.8010062 | embassies.net; cidade-brasil.com.br (3.88921, -51.80123) |

In [7]:
# 1. Manual corrections table (one row per physical venue, key = chaves_local)
df_correcoes_manuais = pd.DataFrame([
    {
        'SG_UF': 'ZZ',
        'CD_MUNICIPIO': 99333,
        'NR_ZONA': 1,
        'NR_LOCAL_VOTACAO': 1015,
        'GEO_LAT_MANUAL': 3.8893706,
        'GEO_LON_MANUAL': -51.8010062,
        'GEO_CONSULTA_MANUAL': "Consulado do Brasil em Saint-Georges de l'Oyapock - 15, Rue Jean Cedia, Angle 01 Henri, "
                               "97313 Saint-Georges de l'Oyapock, French Guiana (source: embassies.net)"
    }
])
df_correcoes_manuais['SG_UF'] = df_correcoes_manuais['SG_UF'].astype(str)
df_correcoes_manuais['NR_LOCAL_VOTACAO'] = df_correcoes_manuais['NR_LOCAL_VOTACAO'].astype('Int64')

cols_manual = ['GEO_LAT_MANUAL', 'GEO_LON_MANUAL', 'GEO_CONSULTA_MANUAL']

# 2. Apply corrections to all sections of each venue
df_merged = df_merged.drop(columns=cols_manual, errors='ignore')
df_merged = df_merged.merge(df_correcoes_manuais, on=chaves_local, how='left')

mask_manual = df_merged['GEO_LAT_MANUAL'].notna()
df_merged.loc[mask_manual, 'lat_clean'] = df_merged.loc[mask_manual, 'GEO_LAT_MANUAL']
df_merged.loc[mask_manual, 'lon_clean'] = df_merged.loc[mask_manual, 'GEO_LON_MANUAL']
df_merged.loc[mask_manual, 'GEO_LAT'] = df_merged.loc[mask_manual, 'GEO_LAT_MANUAL']
df_merged.loc[mask_manual, 'GEO_LON'] = df_merged.loc[mask_manual, 'GEO_LON_MANUAL']
df_merged.loc[mask_manual, 'GEO_METODO'] = 'manual_web_search'
df_merged.loc[mask_manual, 'GEO_CONSULTA'] = df_merged.loc[mask_manual, 'GEO_CONSULTA_MANUAL']
df_merged.loc[mask_manual, 'STATUS_GEO_FINAL'] = 'OK: Manually geocoded venue (web search)'
df_merged = df_merged.drop(columns=cols_manual)

print(f"Manual corrections defined: {len(df_correcoes_manuais)} venue(s) | sections updated: {mask_manual.sum():,}")
print(df_merged.loc[mask_manual, ['SG_UF', 'NM_MUNICIPIO', 'NM_LOCAL_VOTACAO', 'NR_SECAO', 'QT_ELEITORES_TOTAL',
                                  'lat_clean', 'lon_clean', 'GEO_METODO']].to_string(index=False))

# 3. Recompute the re-audit summary
coord_valida = (
    df_merged['lat_clean'].notna() & df_merged['lon_clean'].notna()
    & (df_merged['lat_clean'] != -1) & (df_merged['lon_clean'] != -1)
    & df_merged['lat_clean'].between(-90.0, 90.0) & df_merged['lon_clean'].between(-180.0, 180.0)
)

tabela_reauditoria = (
    df_merged.groupby('STATUS_GEO_FINAL')
    .agg(Sections=('NR_SECAO', 'count'), Voters=('QT_ELEITORES_TOTAL', 'sum'))
    .sort_values('Sections', ascending=False)
    .reset_index()
)
tabela_reauditoria['Pct_Sections'] = (tabela_reauditoria['Sections'] / tabela_reauditoria['Sections'].sum() * 100).round(3)
tabela_reauditoria['Pct_Voters'] = (tabela_reauditoria['Voters'] / tabela_reauditoria['Voters'].sum() * 100).round(3)

print("\n=== GEOGRAPHIC RE-AUDIT SUMMARY (AFTER MANUAL CORRECTIONS) ===")
print(tabela_reauditoria.to_string(index=False))

print("\n=== FIRST AUDIT (rows) x GEO_METODO (columns) ===")
print(pd.crosstab(df_merged['STATUS_GEO'], df_merged['GEO_METODO'], margins=True).to_string())

# 4. Rebuild the global map dataset
df_mapa_global = df_merged[coord_valida & (df_merged['_merge'] == 'both')].copy()
print(f"\nSections available for global map: {len(df_mapa_global):,} of {len(df_merged):,} "
      f"({len(df_mapa_global) / len(df_merged) * 100:.3f}%)")
print(f"Registered voters covered: {df_mapa_global['QT_ELEITORES_TOTAL'].sum():,} of {df_merged['QT_ELEITORES_TOTAL'].sum():,}")

# 5. Re-export re-audit tables
df_merged[~df_merged['STATUS_GEO_FINAL'].isin(['OK: TSE original coordinate', 'OK: Aggregated section (TSE coordinate)'])].to_csv(
    caminho_reauditoria, sep=';', index=False, encoding='utf-8'
)
df_nao_geocodificados = df_merged[df_merged['STATUS_GEO_FINAL'] == 'Error: Unresolved coordinate (excluded from map)'].drop_duplicates(subset=chaves_local)
df_nao_geocodificados.to_csv(caminho_pendentes, sep=';', index=False, encoding='utf-8')

print(f"\nRe-audit report updated: {caminho_reauditoria}")
print(f"Unresolved venues remaining: {len(df_nao_geocodificados):,} (saved to {caminho_pendentes})")

Manual corrections defined: 1 venue(s) | sections updated: 1
SG_UF           NM_MUNICIPIO       NM_LOCAL_VOTACAO  NR_SECAO  QT_ELEITORES_TOTAL  lat_clean  lon_clean        GEO_METODO
   ZZ ST GEORGES DE LOYAPOCK ST GEORGES DE LOYAPOCK      3188                  10   3.889371 -51.801006 manual_web_search

=== GEOGRAPHIC RE-AUDIT SUMMARY (AFTER MANUAL CORRECTIONS) ===
                                   STATUS_GEO_FINAL  Sections    Voters  Pct_Sections  Pct_Voters
                        OK: TSE original coordinate    496015 155266239        95.908      97.808
            OK: Aggregated section (TSE coordinate)     16285   2028512         3.149       1.278
                     Approx: Overseas city centroid      1891    634410         0.366       0.400
           Approx: Brazil CEP/municipality centroid      1596    376383         0.309       0.237
                 OK: Geocoded overseas venue (name)       823    284456         0.159       0.179
           OK: Geocoded Brazil venue (name/

## 5.3 Validation of Geocoded Coordinates
Check the coordinates produced by the automatic cascade before using them:

1. **Brazil:** every venue with `GEO_METODO` in `nominatim_venue_name`, `nominatim_address`, `cep_centroid` or `municipality_centroid` must fall inside its own municipality. TSE municipality names are linked to IBGE codes by normalized name and UF, and points are tested against the `geobr` municipality polygons in SIRGAS 2000 / Brazil Polyconic (EPSG:5880), with a 2 km tolerance for boundary generalization. A rejected point is replaced by the municipal seat (`geobr.read_municipal_seat`), or by a representative point of the polygon when the municipality has no seat record.
2. **Overseas (ZZ):** every venue matched by name (`nominatim_venue_name`) must lie within 50 km (haversine distance) of the centroid of its city, which is retrieved from Nominatim and cached in `cache_centroides_cidades_exterior.csv`. A rejected point is replaced by the city centroid.

The result for each venue is stored in `GEO_VALIDACAO` (`valid`, `rejected` or `not_validated`), with the distance in `GEO_DIST_KM`. The method that existed before validation is kept in `GEO_METODO_ORIGINAL`. Rejected venues receive `GEO_METODO = municipality_centroid`. Run this cell once after 5.2: on a rerun, the replaced points are validated again and their `rejected` label is overwritten.

In [8]:
import geopandas as gpd
import geobr

ano_malha = 2022
limite_tolerancia_km = 2.0
limite_exterior_km = 50.0
metodos_validar_br = ['nominatim_venue_name', 'nominatim_address', 'cep_centroid', 'municipality_centroid']

if 'GEO_METODO_ORIGINAL' not in df_merged.columns:
    df_merged['GEO_METODO_ORIGINAL'] = df_merged['GEO_METODO']

# ===== Part A: Brazil - point inside own municipality =====

# A.1 Municipality polygons and municipal seats (IBGE via geobr)
gdf_mun = geobr.read_municipality(code_muni='all', year=ano_malha)
gdf_mun = gdf_mun[gdf_mun['code_muni'].notna()].copy()
gdf_mun['code_muni'] = gdf_mun['code_muni'].astype(int)
gdf_mun['SG_UF'] = gdf_mun['abbrev_state'].str.upper().str.strip()
gdf_mun['NOME_NORM'] = (
    gdf_mun['name_muni'].str.upper().str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii')
    .str.replace(r"[^A-Z0-9 ]", ' ', regex=True).str.split().str.join(' ')
)

gdf_sedes = geobr.read_municipal_seat(year=2010)
gdf_sedes = gdf_sedes[gdf_sedes['code_muni'].notna()].copy()
gdf_sedes['code_muni'] = gdf_sedes['code_muni'].astype(int)
sedes = gdf_sedes.drop_duplicates('code_muni').set_index('code_muni').geometry.to_crs('EPSG:4674')
pontos_representativos = gdf_mun.set_index('code_muni').to_crs('EPSG:5880').geometry.representative_point().to_crs('EPSG:4674')

# A.2 Crosswalk TSE municipality code -> IBGE code (normalized name + UF)
df_mun_tse = df_merged.loc[df_merged['SG_UF'] != 'ZZ', ['SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO']].drop_duplicates(subset=['SG_UF', 'CD_MUNICIPIO']).copy()
df_mun_tse['NOME_NORM'] = (
    df_mun_tse['NM_MUNICIPIO'].astype(str).str.upper().str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii')
    .str.replace(r"[^A-Z0-9 ]", ' ', regex=True).str.split().str.join(' ')
)
df_cw = df_mun_tse.merge(gdf_mun[['SG_UF', 'NOME_NORM', 'code_muni']].drop_duplicates(['SG_UF', 'NOME_NORM']), on=['SG_UF', 'NOME_NORM'], how='left')
print(f"TSE municipalities: {len(df_cw):,} | linked to IBGE code: {df_cw['code_muni'].notna().sum():,} | not linked: {df_cw['code_muni'].isna().sum():,}")

# A.3 Venues to validate (one row per physical venue)
df_pts_br = (
    df_merged[(df_merged['SG_UF'] != 'ZZ') & df_merged['GEO_METODO'].isin(metodos_validar_br)]
    .drop_duplicates(subset=chaves_local)
    [chaves_local + ['NM_MUNICIPIO', 'NM_LOCAL_VOTACAO', 'GEO_METODO', 'lat_clean', 'lon_clean']]
    .merge(df_cw[['SG_UF', 'CD_MUNICIPIO', 'code_muni']], on=['SG_UF', 'CD_MUNICIPIO'], how='left')
    .reset_index(drop=True)
)

gdf_pts_br = gpd.GeoDataFrame(df_pts_br, geometry=gpd.points_from_xy(df_pts_br['lon_clean'], df_pts_br['lat_clean']), crs='EPSG:4674')
gdf_pts_proj = gdf_pts_br.to_crs('EPSG:5880')
gdf_mun_proj = gdf_mun.drop_duplicates('code_muni').set_index('code_muni').to_crs('EPSG:5880')

tem_poligono = gdf_pts_br['code_muni'].notna() & gdf_pts_br['code_muni'].isin(gdf_mun_proj.index)
codigos_validos = gdf_pts_br.loc[tem_poligono, 'code_muni'].astype(int)
poligonos_proprios = gpd.GeoSeries(gdf_mun_proj.loc[codigos_validos, 'geometry'].values, index=codigos_validos.index, crs='EPSG:5880')

gdf_pts_br['GEO_DIST_KM'] = np.nan
gdf_pts_br.loc[tem_poligono, 'GEO_DIST_KM'] = (gdf_pts_proj.loc[tem_poligono].geometry.distance(poligonos_proprios) / 1000).round(3)

gdf_pts_br['GEO_VALIDACAO'] = np.select(
    [~tem_poligono, gdf_pts_br['GEO_DIST_KM'] <= limite_tolerancia_km],
    ['not_validated: municipality not linked to geobr', 'valid: inside own municipality'],
    default='rejected: outside own municipality'
)

# A.4 Replacement of rejected points: municipal seat, or representative point of the polygon
mask_rej_br = gdf_pts_br['GEO_VALIDACAO'].str.startswith('rejected')
codigos_rej = gdf_pts_br.loc[mask_rej_br, 'code_muni'].astype(int).tolist()
pontos_ref = [sedes.loc[c] if c in sedes.index else pontos_representativos.loc[c] for c in codigos_rej]
origem_ref = ['geobr municipal seat' if c in sedes.index else 'geobr polygon representative point' for c in codigos_rej]

gdf_pts_br['NEW_LAT'] = np.nan
gdf_pts_br['NEW_LON'] = np.nan
gdf_pts_br['NEW_CONSULTA'] = ''
gdf_pts_br.loc[mask_rej_br, 'NEW_LAT'] = [p.y for p in pontos_ref]
gdf_pts_br.loc[mask_rej_br, 'NEW_LON'] = [p.x for p in pontos_ref]
gdf_pts_br.loc[mask_rej_br, 'NEW_CONSULTA'] = [f"{o} (IBGE {c}) after rejection" for o, c in zip(origem_ref, codigos_rej)]
gdf_pts_br['NEW_STATUS'] = np.where(mask_rej_br, 'Approx: Brazil CEP/municipality centroid', '')

print("\n=== BRAZIL: VALIDATION BY METHOD (venues) ===")
print(pd.crosstab(gdf_pts_br['GEO_METODO'], gdf_pts_br['GEO_VALIDACAO'], margins=True).to_string())

# ===== Part B: Overseas (ZZ) - distance to city centroid =====

# B.1 Venues matched by name
df_pts_zz = (
    df_merged[(df_merged['SG_UF'] == 'ZZ') & (df_merged['GEO_METODO'] == 'nominatim_venue_name')]
    .drop_duplicates(subset=chaves_local)
    [chaves_local + ['NM_MUNICIPIO', 'NM_LOCAL_VOTACAO', 'GEO_METODO', 'lat_clean', 'lon_clean']]
    .reset_index(drop=True)
)

# B.2 City centroids (Nominatim, cached on disk)
caminho_cache_cidades = os.path.join(base_dir, "cache_centroides_cidades_exterior.csv")
if not os.path.exists(caminho_cache_cidades):
    pd.DataFrame(columns=['NM_MUNICIPIO', 'CIDADE_LAT', 'CIDADE_LON', 'CIDADE_NOME_OSM']).to_csv(caminho_cache_cidades, sep=';', index=False, encoding='utf-8')

df_cidades = pd.read_csv(caminho_cache_cidades, sep=';', encoding='utf-8')
cidades_pendentes = sorted(set(df_pts_zz['NM_MUNICIPIO'].dropna()) - set(df_cidades['NM_MUNICIPIO']))
print(f"\nOverseas venues matched by name: {len(df_pts_zz):,} | cities: {df_pts_zz['NM_MUNICIPIO'].nunique():,} | pending city lookups: {len(cidades_pendentes):,}")

for cidade in cidades_pendentes:
    params = {'q': cidade, 'format': 'jsonv2', 'limit': 1, 'featuretype': 'settlement'}
    r = sessao.get(url_nominatim, params=params, timeout=30)
    time.sleep(1.1)  # Nominatim usage policy: max 1 request/second
    resultados = r.json() if r.status_code == 200 else []
    pd.DataFrame([{
        'NM_MUNICIPIO': cidade,
        'CIDADE_LAT': float(resultados[0]['lat']) if resultados else np.nan,
        'CIDADE_LON': float(resultados[0]['lon']) if resultados else np.nan,
        'CIDADE_NOME_OSM': resultados[0].get('display_name', '') if resultados else ''
    }]).to_csv(caminho_cache_cidades, sep=';', mode='a', header=False, index=False, encoding='utf-8')

df_cidades = pd.read_csv(caminho_cache_cidades, sep=';', encoding='utf-8').drop_duplicates('NM_MUNICIPIO', keep='last')
df_pts_zz = df_pts_zz.merge(df_cidades, on='NM_MUNICIPIO', how='left')

# B.3 Haversine distance (km) between venue and city centroid
lat1, lon1 = np.radians(df_pts_zz['lat_clean'].astype(float)), np.radians(df_pts_zz['lon_clean'].astype(float))
lat2, lon2 = np.radians(df_pts_zz['CIDADE_LAT'].astype(float)), np.radians(df_pts_zz['CIDADE_LON'].astype(float))
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
df_pts_zz['GEO_DIST_KM'] = (2 * 6371.0088 * np.arcsin(np.sqrt(a))).round(3)

df_pts_zz['GEO_VALIDACAO'] = np.select(
    [df_pts_zz['CIDADE_LAT'].isna(), df_pts_zz['GEO_DIST_KM'] <= limite_exterior_km],
    ['not_validated: city centroid not found', f'valid: within {limite_exterior_km:.0f} km of city centroid'],
    default=f'rejected: more than {limite_exterior_km:.0f} km from city centroid'
)

mask_rej_zz = df_pts_zz['GEO_VALIDACAO'].str.startswith('rejected')
df_pts_zz['NEW_LAT'] = np.where(mask_rej_zz, df_pts_zz['CIDADE_LAT'], np.nan)
df_pts_zz['NEW_LON'] = np.where(mask_rej_zz, df_pts_zz['CIDADE_LON'], np.nan)
df_pts_zz['NEW_CONSULTA'] = np.where(mask_rej_zz, 'Nominatim city centroid after rejection: ' + df_pts_zz['NM_MUNICIPIO'].astype(str), '')
df_pts_zz['NEW_STATUS'] = np.where(mask_rej_zz, 'Approx: Overseas city centroid', '')

print("\n=== OVERSEAS: VALIDATION (venues) ===")
print(df_pts_zz['GEO_VALIDACAO'].value_counts().to_string())

# ===== Part C: Apply validation to all sections =====
cols_validacao = chaves_local + ['GEO_VALIDACAO', 'GEO_DIST_KM', 'NEW_LAT', 'NEW_LON', 'NEW_CONSULTA', 'NEW_STATUS']
df_validacao = pd.concat([pd.DataFrame(gdf_pts_br.drop(columns='geometry'))[cols_validacao], df_pts_zz[cols_validacao]], ignore_index=True)

df_merged = df_merged.drop(columns=['GEO_VALIDACAO', 'GEO_DIST_KM'], errors='ignore')
df_merged = df_merged.merge(df_validacao, on=chaves_local, how='left')

mask_rej = df_merged['NEW_LAT'].notna()
df_merged.loc[mask_rej, 'lat_clean'] = df_merged.loc[mask_rej, 'NEW_LAT']
df_merged.loc[mask_rej, 'lon_clean'] = df_merged.loc[mask_rej, 'NEW_LON']
df_merged.loc[mask_rej, 'GEO_LAT'] = df_merged.loc[mask_rej, 'NEW_LAT']
df_merged.loc[mask_rej, 'GEO_LON'] = df_merged.loc[mask_rej, 'NEW_LON']
df_merged.loc[mask_rej, 'GEO_METODO'] = 'municipality_centroid'
df_merged.loc[mask_rej, 'GEO_CONSULTA'] = df_merged.loc[mask_rej, 'NEW_CONSULTA']
df_merged.loc[mask_rej, 'STATUS_GEO_FINAL'] = df_merged.loc[mask_rej, 'NEW_STATUS']
df_merged['GEO_VALIDACAO'] = df_merged['GEO_VALIDACAO'].fillna('not_applicable')
df_merged = df_merged.drop(columns=['NEW_LAT', 'NEW_LON', 'NEW_CONSULTA', 'NEW_STATUS'])

print(f"\nSections whose coordinate was replaced after rejection: {mask_rej.sum():,}")

# Rejected venues (largest distances first)
df_rejeitados = df_validacao[df_validacao['GEO_VALIDACAO'].str.startswith('rejected')].merge(
    df_merged.drop_duplicates(subset=chaves_local)[chaves_local + ['NM_MUNICIPIO', 'NM_LOCAL_VOTACAO', 'GEO_METODO_ORIGINAL']],
    on=chaves_local, how='left'
).sort_values('GEO_DIST_KM', ascending=False)
print("\n=== REJECTED VENUES (top 20 by distance) ===")
print(df_rejeitados[['SG_UF', 'NM_MUNICIPIO', 'NM_LOCAL_VOTACAO', 'GEO_METODO_ORIGINAL', 'GEO_DIST_KM']].head(20).to_string(index=False))

caminho_validacao = os.path.join(base_dir, "tabela_validacao_geocodificacao.csv")
df_validacao.drop(columns=['NEW_LAT', 'NEW_LON', 'NEW_CONSULTA', 'NEW_STATUS']).merge(
    df_merged.drop_duplicates(subset=chaves_local)[chaves_local + ['NM_MUNICIPIO', 'NM_LOCAL_VOTACAO', 'GEO_METODO_ORIGINAL', 'GEO_METODO', 'lat_clean', 'lon_clean']],
    on=chaves_local, how='left'
).to_csv(caminho_validacao, sep=';', index=False, encoding='utf-8')
print(f"\nValidation report saved to: {caminho_validacao}")

# ===== Part D: Recompute re-audit summary and global map dataset =====
coord_valida = (
    df_merged['lat_clean'].notna() & df_merged['lon_clean'].notna()
    & (df_merged['lat_clean'] != -1) & (df_merged['lon_clean'] != -1)
    & df_merged['lat_clean'].between(-90.0, 90.0) & df_merged['lon_clean'].between(-180.0, 180.0)
)

tabela_reauditoria = (
    df_merged.groupby('STATUS_GEO_FINAL')
    .agg(Sections=('NR_SECAO', 'count'), Voters=('QT_ELEITORES_TOTAL', 'sum'))
    .sort_values('Sections', ascending=False)
    .reset_index()
)
tabela_reauditoria['Pct_Sections'] = (tabela_reauditoria['Sections'] / tabela_reauditoria['Sections'].sum() * 100).round(3)
tabela_reauditoria['Pct_Voters'] = (tabela_reauditoria['Voters'] / tabela_reauditoria['Voters'].sum() * 100).round(3)

print("\n=== GEOGRAPHIC RE-AUDIT SUMMARY (AFTER VALIDATION) ===")
print(tabela_reauditoria.to_string(index=False))

df_mapa_global = df_merged[coord_valida & (df_merged['_merge'] == 'both')].copy()
print(f"\nSections available for global map: {len(df_mapa_global):,} of {len(df_merged):,}")

df_merged[~df_merged['STATUS_GEO_FINAL'].isin(['OK: TSE original coordinate', 'OK: Aggregated section (TSE coordinate)'])].to_csv(
    caminho_reauditoria, sep=';', index=False, encoding='utf-8'
)
print(f"Re-audit report updated: {caminho_reauditoria}")

TSE municipalities: 5,571 | linked to IBGE code: 5,556 | not linked: 15

=== BRAZIL: VALIDATION BY METHOD (venues) ===
GEO_VALIDACAO          not_validated: municipality not linked to geobr  rejected: outside own municipality  valid: inside own municipality  All
GEO_METODO                                                                                                                                     
cep_centroid                                                         2                                  13                             630  645
municipality_centroid                                                0                                   0                              32   32
nominatim_address                                                    0                                   0                              80   80
nominatim_venue_name                                                 0                                   0                              79   79
All              

## 5.4 Refinement of CEP and Municipality Centroids with `geocodebr`
Venues in Brazil that still carry an approximate coordinate (`cep_centroid` or `municipality_centroid`, including venues demoted in 5.3) are geocoded again with the R package `geocodebr` (IPEA), called through `rpy2`. `geocodebr` matches the address against the IBGE National Address File for Statistical Purposes (CNEFE) offline, using UF, municipality, street (`DS_ENDERECO`, split into street and number), CEP (`NR_CEP`) and locality (`NM_BAIRRO`). The first run downloads the CNEFE data to the local `geocodebr` cache.

Each `geocodebr` result is accepted according to its `precisao`:

| `precisao` | New `GEO_METODO` | `STATUS_GEO_FINAL` |
|---|---|---|
| `numero`, `numero_aproximado`, `logradouro` | `geocodebr_street` | OK: Geocoded Brazil venue (geocodebr street/number) |
| `localidade` | `geocodebr_locality` | Approx: Brazil locality centroid (geocodebr) |
| `cep` | `cep_centroid` (CNEFE-based CEP centroid replaces the previous one) | Approx: Brazil CEP/municipality centroid |
| `municipio` or no match | unchanged | unchanged |

`GEO_CONSULTA` records the precision, the matched address and the deviation (`desvio_metros`) reported by `geocodebr`.

In [9]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

metodos_aproximados_br = ['cep_centroid', 'municipality_centroid']

if 'GEO_METODO_ORIGINAL' not in df_merged.columns:
    df_merged['GEO_METODO_ORIGINAL'] = df_merged['GEO_METODO']

# 1. Queue: Brazilian venues with approximate coordinates
df_fila_cbr = (
    df_merged[(df_merged['SG_UF'] != 'ZZ') & df_merged['GEO_METODO'].isin(metodos_aproximados_br)]
    .drop_duplicates(subset=chaves_local)
    [chaves_local + ['NM_MUNICIPIO', 'NM_LOCAL_VOTACAO', 'DS_ENDERECO', 'NM_BAIRRO', 'NR_CEP', 'GEO_METODO']]
    .reset_index(drop=True)
)
df_fila_cbr['id_local'] = np.arange(len(df_fila_cbr), dtype=int)
print(f"Venues sent to geocodebr: {len(df_fila_cbr):,}")
print(df_fila_cbr['GEO_METODO'].value_counts().to_string())

# 2. Split DS_ENDERECO into street and number (number only when preceded by a comma)
endereco = df_fila_cbr['DS_ENDERECO'].fillna('').astype(str).str.strip().str.replace(r'\s*,?\s*S/?N\.?\s*$', '', regex=True, case=False)
partes = endereco.str.extract(r'^(?P<logradouro>.*?)\s*,\s*(?:N[º°O]?\.?\s*)?(?P<numero>\d+)\s*$')
df_fila_cbr['logradouro'] = partes['logradouro'].fillna(endereco).str.strip(' ,')
df_fila_cbr['numero'] = partes['numero'].fillna('')

df_entrada_cbr = pd.DataFrame({
    'id_local': df_fila_cbr['id_local'].astype(int),
    'estado': df_fila_cbr['SG_UF'].astype(str),
    'municipio': df_fila_cbr['NM_MUNICIPIO'].fillna('').astype(str),
    'logradouro': df_fila_cbr['logradouro'].astype(str),
    'numero': df_fila_cbr['numero'].astype(str),
    'cep': df_fila_cbr['NR_CEP'].fillna('').astype(str),
    'localidade': df_fila_cbr['NM_BAIRRO'].fillna('').astype(str)
})

# 3. Run geocodebr in R
with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv['enderecos_py'] = ro.conversion.py2rpy(df_entrada_cbr)

ro.r('''
suppressPackageStartupMessages(library(geocodebr))
enderecos_py <- as.data.frame(enderecos_py)
for (col in c("logradouro", "numero", "cep", "localidade")) {
    enderecos_py[[col]][enderecos_py[[col]] == ""] <- NA
}
enderecos_py$numero <- suppressWarnings(as.integer(enderecos_py$numero))

campos <- geocodebr::definir_campos(
    estado = "estado",
    municipio = "municipio",
    logradouro = "logradouro",
    numero = "numero",
    cep = "cep",
    localidade = "localidade"
)

resultado_cbr <- geocodebr::geocode(
    enderecos = enderecos_py,
    campos_endereco = campos,
    resultado_completo = TRUE,
    resolver_empates = TRUE,
    resultado_sf = FALSE,
    verboso = TRUE
)
resultado_cbr <- as.data.frame(resultado_cbr)
cols_saida <- intersect(c("id_local", "lat", "lon", "precisao", "tipo_resultado", "desvio_metros", "endereco_encontrado"), names(resultado_cbr))
resultado_cbr <- resultado_cbr[, cols_saida]
''')

versao_geocodebr = ro.r('as.character(packageVersion("geocodebr"))')[0]
print(f"geocodebr version: {versao_geocodebr}")

with localconverter(ro.default_converter + pandas2ri.converter):
    df_res_cbr = ro.conversion.rpy2py(ro.globalenv['resultado_cbr'])

df_res_cbr = pd.DataFrame(df_res_cbr)
df_res_cbr['id_local'] = df_res_cbr['id_local'].astype(int)
df_res_cbr = df_res_cbr.drop_duplicates('id_local')
df_fila_cbr = df_fila_cbr.merge(df_res_cbr, on='id_local', how='left')

print("\n=== geocodebr PRECISION BY PREVIOUS METHOD (venues) ===")
print(pd.crosstab(df_fila_cbr['GEO_METODO'], df_fila_cbr['precisao'].fillna('no_match'), margins=True).to_string())

# 4. Acceptance rules by precision
precisao_rua = ['numero', 'numero_aproximado', 'logradouro']
tem_coord = df_fila_cbr['lat'].notna() & df_fila_cbr['lon'].notna()

condicoes_cbr = [
    tem_coord & df_fila_cbr['precisao'].isin(precisao_rua),
    tem_coord & (df_fila_cbr['precisao'] == 'localidade'),
    tem_coord & (df_fila_cbr['precisao'] == 'cep')
]
df_fila_cbr['NEW_METODO'] = np.select(condicoes_cbr, ['geocodebr_street', 'geocodebr_locality', 'cep_centroid'], default='')
df_fila_cbr['NEW_STATUS'] = np.select(
    condicoes_cbr,
    ['OK: Geocoded Brazil venue (geocodebr street/number)',
     'Approx: Brazil locality centroid (geocodebr)',
     'Approx: Brazil CEP/municipality centroid'],
    default=''
)
df_fila_cbr['NEW_CONSULTA'] = (
    'geocodebr ' + df_fila_cbr['precisao'].fillna('').astype(str)
    + ' | ' + df_fila_cbr['endereco_encontrado'].fillna('').astype(str)
    + ' | desvio_m=' + df_fila_cbr['desvio_metros'].round(0).astype(str)
)

df_aceitos_cbr = df_fila_cbr[df_fila_cbr['NEW_METODO'] != ''][chaves_local + ['lat', 'lon', 'NEW_METODO', 'NEW_STATUS', 'NEW_CONSULTA']].rename(
    columns={'lat': 'NEW_LAT', 'lon': 'NEW_LON'}
)
print(f"\nVenues with coordinate replaced by geocodebr: {len(df_aceitos_cbr):,} of {len(df_fila_cbr):,}")
print(df_aceitos_cbr['NEW_METODO'].value_counts().to_string())

# 5. Apply accepted results to all sections of each venue
df_merged = df_merged.merge(df_aceitos_cbr, on=chaves_local, how='left')

mask_cbr = df_merged['NEW_LAT'].notna()
df_merged.loc[mask_cbr, 'lat_clean'] = df_merged.loc[mask_cbr, 'NEW_LAT'].astype(float)
df_merged.loc[mask_cbr, 'lon_clean'] = df_merged.loc[mask_cbr, 'NEW_LON'].astype(float)
df_merged.loc[mask_cbr, 'GEO_LAT'] = df_merged.loc[mask_cbr, 'NEW_LAT'].astype(float)
df_merged.loc[mask_cbr, 'GEO_LON'] = df_merged.loc[mask_cbr, 'NEW_LON'].astype(float)
df_merged.loc[mask_cbr, 'GEO_METODO'] = df_merged.loc[mask_cbr, 'NEW_METODO']
df_merged.loc[mask_cbr, 'GEO_CONSULTA'] = df_merged.loc[mask_cbr, 'NEW_CONSULTA']
df_merged.loc[mask_cbr, 'STATUS_GEO_FINAL'] = df_merged.loc[mask_cbr, 'NEW_STATUS']
df_merged = df_merged.drop(columns=['NEW_LAT', 'NEW_LON', 'NEW_METODO', 'NEW_STATUS', 'NEW_CONSULTA'])

print(f"Sections updated: {mask_cbr.sum():,}")

caminho_geocodebr = os.path.join(base_dir, "tabela_geocodebr_locais.csv")
df_fila_cbr.to_csv(caminho_geocodebr, sep=';', index=False, encoding='utf-8')
print(f"geocodebr report saved to: {caminho_geocodebr}")

# 6. Recompute re-audit summary and global map dataset
coord_valida = (
    df_merged['lat_clean'].notna() & df_merged['lon_clean'].notna()
    & (df_merged['lat_clean'] != -1) & (df_merged['lon_clean'] != -1)
    & df_merged['lat_clean'].between(-90.0, 90.0) & df_merged['lon_clean'].between(-180.0, 180.0)
)

tabela_reauditoria = (
    df_merged.groupby('STATUS_GEO_FINAL')
    .agg(Sections=('NR_SECAO', 'count'), Voters=('QT_ELEITORES_TOTAL', 'sum'))
    .sort_values('Sections', ascending=False)
    .reset_index()
)
tabela_reauditoria['Pct_Sections'] = (tabela_reauditoria['Sections'] / tabela_reauditoria['Sections'].sum() * 100).round(3)
tabela_reauditoria['Pct_Voters'] = (tabela_reauditoria['Voters'] / tabela_reauditoria['Voters'].sum() * 100).round(3)

print("\n=== GEOGRAPHIC RE-AUDIT SUMMARY (AFTER geocodebr) ===")
print(tabela_reauditoria.to_string(index=False))

print("\n=== ORIGINAL METHOD (rows) x FINAL METHOD (columns) ===")
print(pd.crosstab(df_merged['GEO_METODO_ORIGINAL'], df_merged['GEO_METODO'], margins=True).to_string())

df_mapa_global = df_merged[coord_valida & (df_merged['_merge'] == 'both')].copy()
print(f"\nSections available for global map: {len(df_mapa_global):,} of {len(df_merged):,}")

df_merged[~df_merged['STATUS_GEO_FINAL'].isin(['OK: TSE original coordinate', 'OK: Aggregated section (TSE coordinate)'])].to_csv(
    caminho_reauditoria, sep=';', index=False, encoding='utf-8'
)
print(f"Re-audit report updated: {caminho_reauditoria}")

Venues sent to geocodebr: 677
GEO_METODO
cep_centroid             632
municipality_centroid     45
ℹ Padronizando endereços de entrada
ℹ Utilizando dados do CNEFE armazenados localmente
ℹ Geolocalizando endereços
 Casos processados: 0/677 ■                                  0% - dn01 
 Casos processados: 11/677 ■                                  2% - da01 
 Casos processados: 26/677 ■■                                 4% - pn01 
 Casos processados: 29/677 ■■                                 4% - pa01 
 Casos processados: 30/677 ■■                                 4% - dn02 
 Casos processados: 35/677 ■■■                                5% - da02 
 Casos processados: 44/677 ■■■                                6% - pn02 
 Casos processados: 44/677 ■■■                                6% - pa02 
 Casos processados: 46/677 ■■■                                7% - dn03 
 Casos processados: 49/677 ■■■                                7% - da03 
 Casos processados: 51/677 ■■■                            

# 6. Download of Ballot Box Reports (Boletim de Urna - `bweb`) and Presidential Votes per Section
Source: TSE open data, dataset *Resultados 2026 - Boletim de Urna* (`resultados-2026-boletim-de-urna`, source system SISTOT). Each state resource (`UF - Boletim de Urna - Segundo turno - 26.10.2026`) is a ZIP with one CSV (`bweb_2t_<UF>_<timestamp>.csv`) that reproduces the content of every ballot box report (BU): one row per section, office and votable option, with `QT_APTOS`, `QT_COMPARECIMENTO`, `DS_TIPO_VOTAVEL` (`Nominal`, `Branco`, `Nulo`), `NR_VOTAVEL`, `NM_VOTAVEL` and `QT_VOTOS`.

Workflow:
1. list the dataset resources through the portal's CKAN API (`package_show`), keeping only *Boletim de Urna* resources of the target round (`turno_alvo`); the *Hash SHA2* resources are ignored;
2. for each state, download the ZIP and extract the CSV to local disk (`pasta_tmp_bweb`, outside the Google Drive mount);
3. with DuckDB, keep only `DS_CARGO_PERGUNTA = 'Presidente'`, type the columns and write one Parquet file per state in `bweb_2026_<turno>t/`; the ZIP and CSV are then deleted. States that already have a Parquet file are skipped, so the cell resumes after an interruption;
4. aggregate presidential votes per section and identify the two leading candidates.

Aggregated sections are listed in `DS_SECOES_AGREGADAS`; their votes are in the row of the principal section (`NR_SECAO`).

In [11]:
import re

turno_alvo = 2
nome_turno = {1: 'Primeiro turno', 2: 'Segundo turno'}[turno_alvo]

pasta_bweb = os.path.join(base_dir, f"bweb_2026_{turno_alvo}t")        # Parquet output (Google Drive)
pasta_tmp_bweb = f"/home/daves/tmp_bweb_2026_{turno_alvo}t"            # Local disk for ZIP/CSV
os.makedirs(pasta_bweb, exist_ok=True)
os.makedirs(pasta_tmp_bweb, exist_ok=True)

con_bu = duckdb.connect()

# 1. Resource list: CKAN API + dataset HTML page, matched by URL pattern bweb_<turno>t_<UF>_<timestamp>.zip
padrao_bweb = re.compile(rf'https?://[^\s"\'<>]*?bweb_{turno_alvo}t_([A-Z]{{2}})_\d+\.zip', re.IGNORECASE)
cabecalhos = {'User-Agent': 'Mozilla/5.0'}

r_api = requests.get("https://dadosabertos.tse.jus.br/api/3/action/package_show",
                     params={'id': 'resultados-2026-boletim-de-urna'}, headers=cabecalhos, timeout=60)
recursos_api = r_api.json()['result']['resources'] if r_api.status_code == 200 else []
print(f"CKAN API: HTTP {r_api.status_code} | resources listed: {len(recursos_api)}")
print("Sample of resource names returned by the API:")
print("\n".join(f"  {rec.get('name', '')} -> {rec.get('url', '')}" for rec in recursos_api[:6]))

urls_api = [rec.get('url', '') for rec in recursos_api]

r_html = requests.get("https://dadosabertos.tse.jus.br/dataset/resultados-2026-boletim-de-urna", headers=cabecalhos, timeout=60)
urls_html = [m.group(0) for m in padrao_bweb.finditer(r_html.text)] if r_html.status_code == 200 else []
print(f"Dataset HTML page: HTTP {r_html.status_code} | bweb_{turno_alvo}t links found: {len(urls_html)}")

linhas_recursos = []
for url in urls_api + urls_html:
    m = padrao_bweb.search(url)
    if m:
        linhas_recursos.append({'SG_UF': m.group(1).upper(), 'URL': m.group(0), 'ORIGEM': 'api' if url in urls_api else 'html'})

df_recursos = pd.DataFrame(linhas_recursos, columns=['SG_UF', 'URL', 'ORIGEM'])
df_recursos = df_recursos.sort_values(['SG_UF', 'URL']).drop_duplicates('SG_UF', keep='last').reset_index(drop=True)

print(f"\nResources 'Boletim de Urna - {nome_turno}' (bweb_{turno_alvo}t): {len(df_recursos)}")
print(df_recursos.to_string(index=False))

if df_recursos.empty:
    raise ValueError(
        f"No bweb_{turno_alvo}t_<UF>_<timestamp>.zip links found in the CKAN API or in the dataset page. "
        "Copy one 'Ir para recurso' link of a 'Segundo turno' resource and check its file name pattern."
    )

# 2. Download, extract, filter President and save one Parquet per state
for rec in df_recursos.itertuples(index=False):
    uf = rec.SG_UF
    caminho_parquet_uf = os.path.join(pasta_bweb, f"bweb_presidente_{turno_alvo}t_{uf}.parquet")
    if os.path.exists(caminho_parquet_uf):
        print(f"{uf}: Parquet already available, skipped")
        continue

    inicio_uf = time.time()
    caminho_zip = os.path.join(pasta_tmp_bweb, os.path.basename(rec.URL))
    if not os.path.exists(caminho_zip):
        with requests.get(rec.URL, stream=True, timeout=300) as rz:
            rz.raise_for_status()
            with open(caminho_zip + '.part', 'wb') as f:
                for chunk in rz.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
        os.rename(caminho_zip + '.part', caminho_zip)

    with zipfile.ZipFile(caminho_zip, 'r') as z:
        nomes_csv = [n for n in z.namelist() if n.lower().endswith('.csv')]
        for nome in nomes_csv:
            z.extract(nome, pasta_tmp_bweb)
    arquivos_csv_uf = [os.path.join(pasta_tmp_bweb, n) for n in nomes_csv]
    lista_csv_sql = "[" + ", ".join(f"'{f}'" for f in arquivos_csv_uf) + "]"

    con_bu.execute(f'''
    COPY (
        SELECT
            TRY_CAST(NR_TURNO AS INTEGER) AS NR_TURNO,
            UPPER(TRIM(SG_UF)) AS SG_UF,
            TRY_CAST(CD_MUNICIPIO AS INTEGER) AS CD_MUNICIPIO,
            TRIM(NM_MUNICIPIO) AS NM_MUNICIPIO,
            TRY_CAST(NR_ZONA AS INTEGER) AS NR_ZONA,
            TRY_CAST(NR_SECAO AS INTEGER) AS NR_SECAO,
            TRY_CAST(NR_LOCAL_VOTACAO AS INTEGER) AS NR_LOCAL_VOTACAO,
            TRY_CAST(CD_CARGO_PERGUNTA AS INTEGER) AS CD_CARGO_PERGUNTA,
            TRIM(DS_CARGO_PERGUNTA) AS DS_CARGO_PERGUNTA,
            TRY_CAST(QT_APTOS AS INTEGER) AS QT_APTOS,
            TRY_CAST(QT_COMPARECIMENTO AS INTEGER) AS QT_COMPARECIMENTO,
            TRY_CAST(QT_ABSTENCOES AS INTEGER) AS QT_ABSTENCOES,
            TRY_CAST(CD_TIPO_URNA AS INTEGER) AS CD_TIPO_URNA,
            TRIM(DS_TIPO_URNA) AS DS_TIPO_URNA,
            TRY_CAST(CD_TIPO_VOTAVEL AS INTEGER) AS CD_TIPO_VOTAVEL,
            TRIM(DS_TIPO_VOTAVEL) AS DS_TIPO_VOTAVEL,
            TRY_CAST(NR_VOTAVEL AS INTEGER) AS NR_VOTAVEL,
            TRIM(NM_VOTAVEL) AS NM_VOTAVEL,
            TRY_CAST(NR_PARTIDO AS INTEGER) AS NR_PARTIDO,
            TRIM(SG_PARTIDO) AS SG_PARTIDO,
            TRY_CAST(QT_VOTOS AS BIGINT) AS QT_VOTOS,
            TRIM(DS_SECOES_AGREGADAS) AS DS_SECOES_AGREGADAS,
            TRIM(DT_BU_RECEBIDO) AS DT_BU_RECEBIDO
        FROM read_csv({lista_csv_sql}, delim=';', quote='"', encoding='latin-1', header=True, all_varchar=True, union_by_name=True)
        WHERE TRIM(DS_CARGO_PERGUNTA) = 'Presidente'
    ) TO '{caminho_parquet_uf}' (FORMAT PARQUET, COMPRESSION ZSTD)
    ''')

    resumo_uf = con_bu.execute(f'''
        SELECT COUNT(*) AS linhas, COUNT(DISTINCT (CD_MUNICIPIO, NR_ZONA, NR_SECAO)) AS secoes, SUM(QT_VOTOS) AS votos
        FROM read_parquet('{caminho_parquet_uf}')
    ''').fetchone()
    print(f"{uf}: {resumo_uf[1]:,} sections | {resumo_uf[0]:,} rows | {resumo_uf[2]:,} presidential votes | "
          f"{(time.time() - inicio_uf) / 60:.1f} min")

    for f in arquivos_csv_uf:
        os.remove(f)
    os.remove(caminho_zip)

arquivos_parquet_bweb = sorted(glob.glob(os.path.join(pasta_bweb, f"bweb_presidente_{turno_alvo}t_*.parquet")))
ufs_faltantes = sorted(set(df_recursos['SG_UF']) - {os.path.basename(f)[-10:-8] for f in arquivos_parquet_bweb})
print(f"\nState Parquet files available: {len(arquivos_parquet_bweb)} of {len(df_recursos)} | missing: {ufs_faltantes}")

lista_bweb_sql = "[" + ", ".join(f"'{f}'" for f in arquivos_parquet_bweb) + "]"
fonte_bweb = f"read_parquet({lista_bweb_sql})"

# 3. Consistency checks
print("\n=== ROUND / BALLOT BOX TYPE / VOTE TYPE ===")
print(con_bu.execute(f'''
    SELECT NR_TURNO, DS_TIPO_URNA, DS_TIPO_VOTAVEL, COUNT(*) AS linhas, SUM(QT_VOTOS) AS votos
    FROM {fonte_bweb} GROUP BY ALL ORDER BY ALL
''').df().to_string(index=False))

n_inconsistentes = con_bu.execute(f'''
    SELECT COUNT(*) FROM (
        SELECT SG_UF, CD_MUNICIPIO, NR_ZONA, NR_SECAO, MAX(QT_COMPARECIMENTO) AS comparecimento, SUM(QT_VOTOS) AS votos
        FROM {fonte_bweb} GROUP BY ALL HAVING comparecimento <> votos
    )
''').fetchone()[0]
print(f"Sections where presidential votes differ from turnout: {n_inconsistentes:,}")

# 4. Presidential results (nominal votes)
top_candidatos = con_bu.execute(f'''
    SELECT NR_VOTAVEL, NM_VOTAVEL, SG_PARTIDO, SUM(QT_VOTOS) AS total
    FROM {fonte_bweb}
    WHERE DS_TIPO_VOTAVEL = 'Nominal'
    GROUP BY ALL
    ORDER BY total DESC
''').df()
top_candidatos['PCT_VALIDOS'] = (top_candidatos['total'] / top_candidatos['total'].sum() * 100).round(2)
print(f"\n=== PRESIDENTIAL CANDIDATES - {nome_turno.upper()} (BU) ===")
print(top_candidatos.to_string(index=False))

if len(top_candidatos) < 2:
    raise ValueError("Fewer than two presidential candidates with nominal votes.")

cand1 = top_candidatos['NM_VOTAVEL'].iloc[0]
cand2 = top_candidatos['NM_VOTAVEL'].iloc[1]
nr_cand1 = int(top_candidatos['NR_VOTAVEL'].iloc[0])
nr_cand2 = int(top_candidatos['NR_VOTAVEL'].iloc[1])
col_cand1 = 'VOTOS_CAND1'
col_cand2 = 'VOTOS_CAND2'

# 5. Votes per section
df_bu_secao = con_bu.execute(f'''
SELECT
    SG_UF,
    CD_MUNICIPIO,
    NR_ZONA,
    NR_SECAO,
    MAX(QT_APTOS) AS QT_APTOS_BU,
    MAX(QT_COMPARECIMENTO) AS QT_COMPARECIMENTO_BU,
    MAX(DS_SECOES_AGREGADAS) AS DS_SECOES_AGREGADAS,
    SUM(CASE WHEN DS_TIPO_VOTAVEL = 'Nominal' AND NR_VOTAVEL = {nr_cand1} THEN QT_VOTOS ELSE 0 END) AS {col_cand1},
    SUM(CASE WHEN DS_TIPO_VOTAVEL = 'Nominal' AND NR_VOTAVEL = {nr_cand2} THEN QT_VOTOS ELSE 0 END) AS {col_cand2},
    SUM(CASE WHEN DS_TIPO_VOTAVEL = 'Branco' THEN QT_VOTOS ELSE 0 END) AS VOTOS_BRANCOS,
    SUM(CASE WHEN DS_TIPO_VOTAVEL = 'Nulo' THEN QT_VOTOS ELSE 0 END) AS VOTOS_NULOS,
    SUM(QT_VOTOS) AS TOTAL_VOTOS_SECAO
FROM {fonte_bweb}
GROUP BY 1, 2, 3, 4
''').df()

for col in ['CD_MUNICIPIO', 'NR_ZONA', 'NR_SECAO']:
    df_bu_secao[col] = df_bu_secao[col].astype(int)

caminho_bu_secao = os.path.join(base_dir, f"votos_presidente_{turno_alvo}t_secao_bweb.parquet")
con_bu.register('df_bu_secao_tmp', df_bu_secao)
con_bu.execute(f"COPY df_bu_secao_tmp TO '{caminho_bu_secao}' (FORMAT PARQUET, COMPRESSION ZSTD)")
con_bu.unregister('df_bu_secao_tmp')

# 6. Match with the georeferenced section catalog
chaves_bu = set(map(tuple, df_bu_secao[chaves_join].values))
chaves_geo = set(map(tuple, df_mapa_global[chaves_join].values))
print(f"\nTotal sections tallied: {len(df_bu_secao):,} | with coordinates: {len(chaves_bu & chaves_geo):,} | "
      f"without coordinates: {len(chaves_bu - chaves_geo):,}")
print(f"Votes: {df_bu_secao['TOTAL_VOTOS_SECAO'].sum():,} | Turnout: {df_bu_secao['QT_COMPARECIMENTO_BU'].sum():,} | "
      f"Eligible: {df_bu_secao['QT_APTOS_BU'].sum():,}")
print(f"Candidates identified: {cand1} ({nr_cand1}) vs {cand2} ({nr_cand2})")
print(f"Section results saved to: {caminho_bu_secao}")

CKAN API: HTTP 200 | resources listed: 56
Sample of resource names returned by the API:
  AC - Boletim de Urna - Primeiro turno - 05.10.2026 -> https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2026/buweb/bweb_1t_AC_051020261403.zip
  AC - Hash SHA2 - Primeiro turno - 05.10.2026 -> https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2026/buweb/bweb_1t_AC_051020261403.zip.sha512
  AL - Boletim de Urna - Primeiro turno - 05.10.2026 -> https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2026/buweb/bweb_1t_AL_051020261403.zip
  AL - Hash SHA2 - Primeiro turno - 05.10.2026 -> https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2026/buweb/bweb_1t_AL_051020261403.zip.sha512
  AM - Boletim de Urna - Primeiro turno - 05.10.2026 -> https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2026/buweb/bweb_1t_AM_051020261403.zip
  AM - Hash SHA2 - Primeiro turno - 05.10.2026 -> https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2026/buweb/bweb_1t_AM_051020261403.zip.sha5

# 7. Spatial Join and Interactive Global HTML Map with Presidential Results
Join the tallied section results of the target round (`turno_alvo`, section 6) with the re-audited and validated coordinates (`df_mapa_global`, sections 5.2 to 5.4), aggregate by physical voting venue and compute the local winner. Colors: Lula in red, the other contender in blue, ties in gray.

The interactive Leaflet/Folium map has three overlay layers (selectable in the layer control):

1. **Neighborhoods (bairros) - winner:** IBGE neighborhood polygons (`geobr.read_neighborhood`, 2010) colored by the winner among the voting venues located inside each neighborhood (point-in-polygon). Only venues with exact or street-level coordinates are used (`tse_original`, `manual_web_search`, `nominatim_venue_name`, `nominatim_address`, `geocodebr_street`), because centroids would assign votes to arbitrary neighborhoods. IBGE neighborhoods exist only for municipalities with an official neighborhood division, so many areas have no neighborhood polygon.
2. **Municipalities - winner:** municipality polygons (`geobr.read_municipality`, 2022) colored by the winner among all Brazilian venues located inside each municipality, giving full territorial coverage below the neighborhood layer.
3. **Overseas cities - winner:** each overseas city (`SG_UF = ZZ`) is drawn as a circle at the vote-weighted centroid of its voting venues, colored by the winner; the circle area grows linearly with votes cast (radius = 5 km + 150 m × √votes) and the fill opacity follows the same margin rule as the polygons.
4. **Voting venues:** venue markers in `FastMarkerCluster`; solid markers for exact or street-level coordinates, hollow dashed markers for approximate centroids (`cep_centroid`, `geocodebr_locality`, `municipality_centroid`).

Polygon fill opacity grows with the winner's margin over the two-candidate vote (from 0.20 to 0.75 at a margin of 40 p.p. or more). Geometries are simplified in EPSG:5880 (30 m for neighborhoods, 300 m for municipalities) to keep the HTML file light.

In [15]:
import json
import geopandas as gpd
import geobr
from shapely.geometry import shape

# 1. Join re-audited coordinates (after 5.2-5.4) with section results of the target round
df_completo = pd.merge(df_mapa_global, df_bu_secao, on=chaves_join, how='inner')
print(f"Sections with coordinates and round {turno_alvo} votes: {len(df_completo):,}")

# 2. Aggregate by physical voting venue
df_completo[['DS_ENDERECO', 'NM_LOCAL_VOTACAO', 'NM_MUNICIPIO']] = df_completo[['DS_ENDERECO', 'NM_LOCAL_VOTACAO', 'NM_MUNICIPIO']].fillna('')
cols_votos = ['QT_SECOES', col_cand1, col_cand2, 'VOTOS_BRANCOS', 'VOTOS_NULOS', 'TOTAL_VOTOS_SECAO']

df_mapa_locais = df_completo.groupby(
    ['SG_UF', 'NM_MUNICIPIO', 'NM_LOCAL_VOTACAO', 'DS_ENDERECO', 'lat_clean', 'lon_clean', 'GEO_METODO'],
    as_index=False
).agg({
    'NR_SECAO': 'count',
    col_cand1: 'sum',
    col_cand2: 'sum',
    'VOTOS_BRANCOS': 'sum',
    'VOTOS_NULOS': 'sum',
    'TOTAL_VOTOS_SECAO': 'sum'
}).rename(columns={'NR_SECAO': 'QT_SECOES'})

# 3. Candidate colors (Lula in red, the other contender in blue) and venue-level winner
cor_vermelho, cor_azul, cor_empate = '#dc3545', '#0d6efd', '#6c757d'
cor_cand1 = cor_vermelho if 'LULA' in str(cand1).upper() else cor_azul
cor_cand2 = cor_vermelho if 'LULA' in str(cand2).upper() else cor_azul
if cor_cand1 == cor_cand2:
    cor_cand2 = cor_azul if cor_cand1 == cor_vermelho else cor_vermelho
rotulo_empate = 'Empate'
print(f"Colors: {cand1} = {cor_cand1} | {cand2} = {cor_cand2}")

df_mapa_locais['VENCEDOR'] = np.where(
    df_mapa_locais[col_cand1] > df_mapa_locais[col_cand2], cand1,
    np.where(df_mapa_locais[col_cand2] > df_mapa_locais[col_cand1], cand2, rotulo_empate)
)
df_mapa_locais['PCT_CAND1'] = np.where(df_mapa_locais['TOTAL_VOTOS_SECAO'] > 0,
                                       (df_mapa_locais[col_cand1] / df_mapa_locais['TOTAL_VOTOS_SECAO'] * 100).round(1), 0.0)
df_mapa_locais['PCT_CAND2'] = np.where(df_mapa_locais['TOTAL_VOTOS_SECAO'] > 0,
                                       (df_mapa_locais[col_cand2] / df_mapa_locais['TOTAL_VOTOS_SECAO'] * 100).round(1), 0.0)

print(f"Physical venues on the map: {len(df_mapa_locais):,}")
print(df_mapa_locais['GEO_METODO'].value_counts().to_string())

# 4. Base map (default basemap without labels; labeled options in the layer control)
mapa_votos = folium.Map(location=[-10.0, -40.0], zoom_start=3, tiles=None, world_copy_jump=True)

folium.TileLayer(
    tiles='https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> &copy; <a href="https://carto.com/attributions">CARTO</a>',
    name='Mapa base claro (sem rótulos)', subdomains='abcd', max_zoom=20
).add_to(mapa_votos)
folium.TileLayer(
    tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> &copy; <a href="https://carto.com/attributions">CARTO</a>',
    name='Mapa base claro (com rótulos)', subdomains='abcd', max_zoom=20
).add_to(mapa_votos)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, HERE, Garmin, &copy; OpenStreetMap',
    name='Esri cinza claro', max_zoom=16
).add_to(mapa_votos)

# 5. Brazilian polygons: neighborhoods and municipalities
url_bairros_2022 = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/bairros/shp/BR/BR_bairros_CD2022.zip"
ano_municipios = 2022
ano_estados = 2020
tolerancia_bairro_m = 30
tolerancia_municipio_m = 300
tolerancia_estado_m = 500
metodos_precisos = ['tse_original', 'manual_web_search', 'nominatim_venue_name', 'nominatim_address', 'geocodebr_street']

df_locais_br = df_mapa_locais[df_mapa_locais['SG_UF'] != 'ZZ'].copy()
gdf_locais = gpd.GeoDataFrame(df_locais_br, geometry=gpd.points_from_xy(df_locais_br['lon_clean'], df_locais_br['lat_clean']), crs='EPSG:4674')

# IBGE 2022 Census neighborhoods (bairros), downloaded once and read directly from the ZIP
pasta_bairros = os.path.join(base_dir, "ibge_bairros_2022")
os.makedirs(pasta_bairros, exist_ok=True)
caminho_zip_bairros = os.path.join(pasta_bairros, "BR_bairros_CD2022.zip")
if not os.path.exists(caminho_zip_bairros):
    print("Downloading BR_bairros_CD2022.zip from IBGE...")
    with requests.get(url_bairros_2022, stream=True, timeout=600) as rb:
        rb.raise_for_status()
        with open(caminho_zip_bairros + '.part', 'wb') as f:
            for chunk in rb.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
    os.rename(caminho_zip_bairros + '.part', caminho_zip_bairros)

with zipfile.ZipFile(caminho_zip_bairros, 'r') as z:
    nome_shp_bairros = [n for n in z.namelist() if n.lower().endswith('.shp')][0]
gdf_bairros = gpd.read_file(f"zip://{caminho_zip_bairros}!{nome_shp_bairros}")
print(f"IBGE 2022 neighborhoods: {len(gdf_bairros):,} polygons | CRS: {gdf_bairros.crs} | columns: {list(gdf_bairros.columns)}")
gdf_bairros = gdf_bairros.to_crs('EPSG:4674')
gdf_bairros = gdf_bairros[gdf_bairros.geometry.notna() & ~gdf_bairros.geometry.is_empty].reset_index(drop=True)
gdf_bairros['geometry'] = gdf_bairros.geometry.make_valid()
col_uf_bairro = 'SIGLA_UF' if 'SIGLA_UF' in gdf_bairros.columns else 'NM_UF'
gdf_bairros['name_neighborhood'] = gdf_bairros['NM_BAIRRO'].astype(str)
gdf_bairros['name_muni'] = gdf_bairros['NM_MUN'].astype(str)
gdf_bairros['abbrev_state'] = gdf_bairros[col_uf_bairro].astype(str)
gdf_bairros['ID_POLIGONO'] = np.arange(len(gdf_bairros))
pts_bairro = gpd.sjoin(gdf_locais[gdf_locais['GEO_METODO'].isin(metodos_precisos)][cols_votos + ['geometry']],
                       gdf_bairros[['ID_POLIGONO', 'geometry']], how='inner', predicate='within')
agg_bairro = pts_bairro.groupby('ID_POLIGONO').agg(QT_LOCAIS=('QT_SECOES', 'size'), **{c: (c, 'sum') for c in cols_votos}).reset_index()
gdf_bairros_res = gdf_bairros.merge(agg_bairro, on='ID_POLIGONO', how='inner')
gdf_bairros_res['NOME'] = gdf_bairros_res['name_neighborhood'].astype(str) + ' - ' + gdf_bairros_res['name_muni'].astype(str) + '/' + gdf_bairros_res['abbrev_state'].astype(str)

gdf_mun_mapa = geobr.read_municipality(code_muni='all', year=ano_municipios).to_crs('EPSG:4674')
gdf_mun_mapa = gdf_mun_mapa[gdf_mun_mapa.geometry.notna()].reset_index(drop=True)
gdf_mun_mapa['ID_POLIGONO'] = np.arange(len(gdf_mun_mapa))
pts_mun = gpd.sjoin(gdf_locais[cols_votos + ['geometry']], gdf_mun_mapa[['ID_POLIGONO', 'geometry']], how='inner', predicate='within')
agg_mun = pts_mun.groupby('ID_POLIGONO').agg(QT_LOCAIS=('QT_SECOES', 'size'), **{c: (c, 'sum') for c in cols_votos}).reset_index()
gdf_mun_res = gdf_mun_mapa.merge(agg_mun, on='ID_POLIGONO', how='inner')
gdf_mun_res['NOME'] = gdf_mun_res['name_muni'].astype(str) + '/' + gdf_mun_res['abbrev_state'].astype(str)

# 6. Overseas cities: aggregation and OSM city boundaries (Nominatim reverse geocoding, cached)
df_zz = df_mapa_locais[df_mapa_locais['SG_UF'] == 'ZZ'].copy()
df_zz['LAT_POND'] = df_zz['lat_clean'] * df_zz['TOTAL_VOTOS_SECAO']
df_zz['LON_POND'] = df_zz['lon_clean'] * df_zz['TOTAL_VOTOS_SECAO']
df_cidades_zz = df_zz.groupby('NM_MUNICIPIO', as_index=False).agg(
    QT_LOCAIS=('NM_LOCAL_VOTACAO', 'size'), LAT_POND=('LAT_POND', 'sum'), LON_POND=('LON_POND', 'sum'),
    LAT_MEDIA=('lat_clean', 'mean'), LON_MEDIA=('lon_clean', 'mean'), **{c: (c, 'sum') for c in cols_votos}
)
df_cidades_zz['LAT'] = np.where(df_cidades_zz['TOTAL_VOTOS_SECAO'] > 0, df_cidades_zz['LAT_POND'] / df_cidades_zz['TOTAL_VOTOS_SECAO'], df_cidades_zz['LAT_MEDIA'])
df_cidades_zz['LON'] = np.where(df_cidades_zz['TOTAL_VOTOS_SECAO'] > 0, df_cidades_zz['LON_POND'] / df_cidades_zz['TOTAL_VOTOS_SECAO'], df_cidades_zz['LON_MEDIA'])

caminho_cache_poligonos = os.path.join(base_dir, "cache_poligonos_cidades_exterior.csv")
if not os.path.exists(caminho_cache_poligonos):
    pd.DataFrame(columns=['NM_MUNICIPIO', 'OSM_NOME', 'OSM_TIPO', 'GEOJSON']).to_csv(caminho_cache_poligonos, sep=';', index=False, encoding='utf-8')
df_cache_pol = pd.read_csv(caminho_cache_poligonos, sep=';', encoding='utf-8', keep_default_na=False).drop_duplicates('NM_MUNICIPIO', keep='last')
cidades_com_poligono = set(df_cache_pol.loc[df_cache_pol['GEOJSON'] != '', 'NM_MUNICIPIO'])
df_pendentes_pol = df_cidades_zz[~df_cidades_zz['NM_MUNICIPIO'].isin(cidades_com_poligono)]   # also retries cities cached without polygon
print(f"\nOverseas cities: {len(df_cidades_zz):,} | boundaries cached: {len(df_cidades_zz) - len(df_pendentes_pol):,} | pending: {len(df_pendentes_pol):,}")

# Municipal boundary containing the city's venues: city level first (zoom 10), then town/municipality (12), then county/district (8)
for cidade in df_pendentes_pol.itertuples(index=False):
    nome_osm, tipo_osm, geojson_osm = '', '', ''
    for zoom_osm in [10, 12, 8]:
        r = requests.get(
            "https://nominatim.openstreetmap.org/reverse",
            params={'lat': float(cidade.LAT), 'lon': float(cidade.LON), 'zoom': zoom_osm, 'format': 'jsonv2',
                    'polygon_geojson': 1, 'polygon_threshold': 0.001, 'accept-language': 'pt-BR'},
            headers={'User-Agent': 'geografia-do-voto-research/1.0'}, timeout=60
        )
        time.sleep(1.1)  # Nominatim usage policy: max 1 request/second
        dados = r.json() if r.status_code == 200 else {}
        geometria = dados.get('geojson', {}) if isinstance(dados, dict) else {}
        if geometria.get('type') in ('Polygon', 'MultiPolygon'):
            nome_osm = dados.get('name', '')
            tipo_osm = f"{dados.get('addresstype', dados.get('type', ''))} (zoom {zoom_osm})"
            geojson_osm = json.dumps(geometria)
            break
    pd.DataFrame([{'NM_MUNICIPIO': cidade.NM_MUNICIPIO, 'OSM_NOME': nome_osm, 'OSM_TIPO': tipo_osm, 'GEOJSON': geojson_osm}]).to_csv(
        caminho_cache_poligonos, sep=';', mode='a', header=False, index=False, encoding='utf-8')
    print(f"  {cidade.NM_MUNICIPIO}: {'boundary ' + tipo_osm + ' - ' + nome_osm if geojson_osm else 'no boundary polygon in OSM'}")

df_cache_pol = pd.read_csv(caminho_cache_poligonos, sep=';', encoding='utf-8', keep_default_na=False).drop_duplicates('NM_MUNICIPIO', keep='last')
df_cidades_zz = df_cidades_zz.merge(df_cache_pol, on='NM_MUNICIPIO', how='left')
df_cidades_zz['GEOJSON'] = df_cidades_zz['GEOJSON'].fillna('')
df_cidades_zz['OSM_NOME'] = df_cidades_zz['OSM_NOME'].fillna('')

df_zz_pol = df_cidades_zz[df_cidades_zz['GEOJSON'] != ''].copy()
gdf_zz_res = gpd.GeoDataFrame(df_zz_pol, geometry=[shape(json.loads(g)) for g in df_zz_pol['GEOJSON']], crs='EPSG:4326')
gdf_zz_res['NOME'] = gdf_zz_res['NM_MUNICIPIO'].astype(str) + ' (exterior) - ' + gdf_zz_res['OSM_NOME'].astype(str) + ' [' + gdf_zz_res['OSM_TIPO'].astype(str) + ']'
df_zz_circulos = df_cidades_zz[df_cidades_zz['GEOJSON'] == ''].copy()
print(f"Overseas cities painted with OSM boundary: {len(gdf_zz_res):,} | drawn as circles (no boundary in OSM): {len(df_zz_circulos):,}")

# 7. Winner, two-candidate share, margin, color and opacity for every area layer
gdf_camadas = {}
for nome_camada, gdf_res, tolerancia_m in [('bairro', gdf_bairros_res, tolerancia_bairro_m),
                                            ('municipio', gdf_mun_res, tolerancia_municipio_m),
                                            ('exterior', gdf_zz_res, 0)]:
    votos_validos_2 = gdf_res[col_cand1] + gdf_res[col_cand2]
    gdf_res['PCT_CAND1'] = np.where(votos_validos_2 > 0, (gdf_res[col_cand1] / votos_validos_2 * 100).round(1), 0.0)
    gdf_res['PCT_CAND2'] = np.where(votos_validos_2 > 0, (gdf_res[col_cand2] / votos_validos_2 * 100).round(1), 0.0)
    gdf_res['MARGEM_PP'] = (gdf_res['PCT_CAND1'] - gdf_res['PCT_CAND2']).abs().round(1)
    gdf_res['VENCEDOR'] = np.where(gdf_res[col_cand1] > gdf_res[col_cand2], cand1,
                                   np.where(gdf_res[col_cand2] > gdf_res[col_cand1], cand2, rotulo_empate))
    gdf_res['COR'] = np.where(gdf_res['VENCEDOR'] == cand1, cor_cand1, np.where(gdf_res['VENCEDOR'] == cand2, cor_cand2, cor_empate))
    gdf_res['OPACIDADE'] = (0.20 + 0.55 * np.clip(gdf_res['MARGEM_PP'] / 40.0, 0, 1)).round(2)
    if tolerancia_m > 0:
        gdf_res['geometry'] = gdf_res.geometry.to_crs('EPSG:5880').simplify(tolerancia_m).to_crs('EPSG:4326')
    gdf_camadas[nome_camada] = gpd.GeoDataFrame(gdf_res[[
        'NOME', 'QT_LOCAIS', 'QT_SECOES', col_cand1, col_cand2, 'PCT_CAND1', 'PCT_CAND2',
        'VOTOS_BRANCOS', 'VOTOS_NULOS', 'TOTAL_VOTOS_SECAO', 'MARGEM_PP', 'VENCEDOR', 'COR', 'OPACIDADE', 'geometry'
    ]], geometry='geometry').to_crs('EPSG:4326')

votos_br = df_locais_br['TOTAL_VOTOS_SECAO'].sum()
print(f"\nNeighborhood polygons with votes: {len(gdf_camadas['bairro']):,} | votes covered: {gdf_camadas['bairro']['TOTAL_VOTOS_SECAO'].sum() / votos_br * 100:.1f}% of Brazilian votes")
print(f"Municipality polygons with votes: {len(gdf_camadas['municipio']):,} | votes covered: {gdf_camadas['municipio']['TOTAL_VOTOS_SECAO'].sum() / votos_br * 100:.1f}%")
for nome_camada in ['bairro', 'municipio', 'exterior']:
    print(f"Areas won ({nome_camada}): {gdf_camadas[nome_camada]['VENCEDOR'].value_counts().to_dict()}")

campos_tooltip = ['NOME', 'VENCEDOR', 'PCT_CAND1', 'PCT_CAND2', 'MARGEM_PP', 'TOTAL_VOTOS_SECAO', 'QT_SECOES', 'QT_LOCAIS']
aliases_tooltip = ['Área:', 'Vencedor:', f'{cand1} (% dos votos nos 2 candidatos):', f'{cand2} (% dos votos nos 2 candidatos):',
                   'Margem (p.p.):', 'Votos apurados:', 'Seções:', 'Locais de votação:']

folium.GeoJson(
    gdf_camadas['municipio'], name='Municípios - vencedor',
    style_function=lambda f: {'fillColor': f['properties']['COR'], 'color': '#ffffff', 'weight': 0.3, 'fillOpacity': f['properties']['OPACIDADE'] * 0.6},
    highlight_function=lambda f: {'weight': 2, 'color': '#333333'},
    tooltip=folium.GeoJsonTooltip(fields=campos_tooltip, aliases=aliases_tooltip, localize=True, sticky=True)
).add_to(mapa_votos)

folium.GeoJson(
    gdf_camadas['bairro'], name='Bairros - vencedor',
    style_function=lambda f: {'fillColor': f['properties']['COR'], 'color': '#555555', 'weight': 0.4, 'fillOpacity': f['properties']['OPACIDADE']},
    highlight_function=lambda f: {'weight': 2, 'color': '#000000'},
    tooltip=folium.GeoJsonTooltip(fields=campos_tooltip, aliases=aliases_tooltip, localize=True, sticky=True)
).add_to(mapa_votos)

# 8. State boundaries (lines only, so they do not block the tooltips of the polygons below)
gdf_estados = geobr.read_state(code_state='all', year=ano_estados).to_crs('EPSG:5880')
gdf_estados['geometry'] = gdf_estados.geometry.simplify(tolerancia_estado_m).boundary
gdf_estados = gdf_estados.to_crs('EPSG:4326')[['name_state', 'abbrev_state', 'geometry']]
folium.GeoJson(
    gdf_estados, name='Limites estaduais',
    style_function=lambda f: {'color': '#222222', 'weight': 1.5, 'opacity': 0.9},
    tooltip=folium.GeoJsonTooltip(fields=['name_state', 'abbrev_state'], aliases=['Estado:', 'UF:'])
).add_to(mapa_votos)

# 9. Overseas cities: OSM boundaries plus circles for cities without a boundary polygon
camada_exterior = folium.FeatureGroup(name='Cidades no exterior - vencedor', show=True)
folium.GeoJson(
    gdf_camadas['exterior'],
    style_function=lambda f: {'fillColor': f['properties']['COR'], 'color': '#555555', 'weight': 0.4, 'fillOpacity': f['properties']['OPACIDADE']},   # same style as Brazilian neighborhoods
    highlight_function=lambda f: {'weight': 2, 'color': '#000000'},
    tooltip=folium.GeoJsonTooltip(fields=campos_tooltip, aliases=aliases_tooltip, localize=True, sticky=True)
).add_to(camada_exterior)

votos_validos_c = df_zz_circulos[col_cand1] + df_zz_circulos[col_cand2]
df_zz_circulos['PCT_CAND1'] = np.where(votos_validos_c > 0, (df_zz_circulos[col_cand1] / votos_validos_c * 100).round(1), 0.0)
df_zz_circulos['PCT_CAND2'] = np.where(votos_validos_c > 0, (df_zz_circulos[col_cand2] / votos_validos_c * 100).round(1), 0.0)
df_zz_circulos['MARGEM_PP'] = (df_zz_circulos['PCT_CAND1'] - df_zz_circulos['PCT_CAND2']).abs().round(1)
df_zz_circulos['VENCEDOR'] = np.where(df_zz_circulos[col_cand1] > df_zz_circulos[col_cand2], cand1,
                                      np.where(df_zz_circulos[col_cand2] > df_zz_circulos[col_cand1], cand2, rotulo_empate))
df_zz_circulos['COR'] = np.where(df_zz_circulos['VENCEDOR'] == cand1, cor_cand1, np.where(df_zz_circulos['VENCEDOR'] == cand2, cor_cand2, cor_empate))
df_zz_circulos['OPACIDADE'] = (0.20 + 0.55 * np.clip(df_zz_circulos['MARGEM_PP'] / 40.0, 0, 1)).round(2)
df_zz_circulos['RAIO_M'] = (5000 + 150 * np.sqrt(df_zz_circulos['TOTAL_VOTOS_SECAO'].clip(lower=0))).round(0)

for cidade in df_zz_circulos.itertuples(index=False):
    tooltip_cidade = (
        f"<b>{cidade.NM_MUNICIPIO}</b> (exterior)<br>"
        f"<b>Vencedor:</b> {cidade.VENCEDOR}<br>"
        f"{cand1}: {getattr(cidade, col_cand1):,} ({cidade.PCT_CAND1}% dos votos nos 2 candidatos)<br>"
        f"{cand2}: {getattr(cidade, col_cand2):,} ({cidade.PCT_CAND2}% dos votos nos 2 candidatos)<br>"
        f"<b>Margem:</b> {cidade.MARGEM_PP} p.p.<br>"
        f"<b>Votos apurados:</b> {cidade.TOTAL_VOTOS_SECAO:,} | <b>Seções:</b> {cidade.QT_SECOES:,} | <b>Locais:</b> {cidade.QT_LOCAIS}"
    ).replace(',', '.')
    folium.Circle(
        location=[float(cidade.LAT), float(cidade.LON)], radius=float(cidade.RAIO_M),
        color=cidade.COR, weight=1, fill=True, fill_color=cidade.COR, fill_opacity=float(cidade.OPACIDADE),
        tooltip=folium.Tooltip(tooltip_cidade, sticky=True)
    ).add_to(camada_exterior)
camada_exterior.add_to(mapa_votos)

# 10. Voting venue markers (on top of all area layers)
cols_cluster = [
    'lat_clean', 'lon_clean', 'NM_LOCAL_VOTACAO', 'NM_MUNICIPIO', 'SG_UF',
    'DS_ENDERECO', 'QT_SECOES', col_cand1, 'PCT_CAND1', col_cand2,
    'PCT_CAND2', 'VOTOS_BRANCOS', 'VOTOS_NULOS', 'TOTAL_VOTOS_SECAO', 'VENCEDOR', 'GEO_METODO'
]
dados_cluster = df_mapa_locais[cols_cluster].astype(object).values.tolist()
dados_cluster = [[v.item() if hasattr(v, 'item') else v for v in linha] for linha in dados_cluster]

cand1_js = cand1.replace("\\", "\\\\").replace("'", "\\'")
cand2_js = cand2.replace("\\", "\\\\").replace("'", "\\'")

callback_leaflet = f'''
function (row) {{
    var cor = row[14] === '{cand1_js}' ? '{cor_cand1}' : (row[14] === '{cand2_js}' ? '{cor_cand2}' : '{cor_empate}');
    var aproximado = (row[15] === 'cep_centroid' || row[15] === 'municipality_centroid' || row[15] === 'geocodebr_locality');
    var fmt = function (x) {{ return Number(x).toLocaleString('pt-BR'); }};
    var marker = L.circleMarker(new L.LatLng(row[0], row[1]), {{
        radius: aproximado ? 5 : 4, color: cor, weight: aproximado ? 1.5 : 1, dashArray: aproximado ? '3' : null,
        fill: true, fillColor: cor, fillOpacity: aproximado ? 0.15 : 0.75
    }});
    var popupContent =
        '<div style="min-width: 240px; font-family: sans-serif; font-size: 12px;">' +
        '<b style="font-size: 13px;">' + row[2] + '</b><br>' +
        '<span style="color: #666;">' + row[3] + ' - ' + (row[4] === 'ZZ' ? 'Exterior' : row[4]) + '</span><br>' +
        '<i>' + row[5] + '</i><br>' +
        '<hr style="margin: 6px 0;">' +
        '<b>Seções no local:</b> ' + fmt(row[6]) + '<br>' +
        '<b>Total de votos:</b> ' + fmt(row[13]) + '<br><br>' +
        '<span style="color: {cor_cand1}; font-weight: bold;">{cand1_js}:</span> ' + fmt(row[7]) + ' (' + fmt(row[8]) + '%)<br>' +
        '<span style="color: {cor_cand2}; font-weight: bold;">{cand2_js}:</span> ' + fmt(row[9]) + ' (' + fmt(row[10]) + '%)<br>' +
        '<b>Brancos:</b> ' + fmt(row[11]) + ' | <b>Nulos:</b> ' + fmt(row[12]) + '<br>' +
        '<hr style="margin: 6px 0;">' +
        '<b>Vencedor no local:</b> ' + row[14] + '<br>' +
        '<b>Origem da coordenada (GEO_METODO):</b> ' + row[15] + (aproximado ? ' <span style="color:#b8860b;">(aproximada)</span>' : '') +
        '</div>';
    marker.bindPopup(popupContent);
    return marker;
}}
'''

cluster = FastMarkerCluster(data=dados_cluster, callback=callback_leaflet, name='Locais de votação')
mapa_votos.add_child(cluster)
folium.LayerControl(collapsed=True).add_to(mapa_votos)

# 11. Legend and Portuguese labels for the Leaflet controls
legenda_html = f'''
<div style="position: fixed; bottom: 25px; left: 25px; z-index: 9999; background: white;
            padding: 10px 12px; border-radius: 6px; box-shadow: 0 1px 4px rgba(0,0,0,0.3);
            font-family: sans-serif; font-size: 12px;">
  <b>Eleição presidencial 2026 - {turno_alvo}º turno - vencedor</b><br>
  <span style="color:{cor_cand1};">&#9679;</span> {cand1}<br>
  <span style="color:{cor_cand2};">&#9679;</span> {cand2}<br>
  <span style="color:{cor_empate};">&#9679;</span> Empate<br>
  <span style="color:#444;">&#9675;</span> Coordenada aproximada (centroide de CEP, localidade ou município)<br>
  <span style="display:inline-block;width:10px;height:10px;background:{cor_cand1};opacity:0.6;"></span>
  <span style="display:inline-block;width:10px;height:10px;background:{cor_cand2};opacity:0.6;"></span>
  Vencedor no bairro, município ou cidade no exterior (cor mais forte = margem maior)<br>
  <span style="display:inline-block;width:14px;height:0;border-top:2px solid #222;vertical-align:middle;"></span>
  Limites estaduais
</div>
'''
mapa_votos.get_root().html.add_child(folium.Element(legenda_html))

script_pt = '''
document.documentElement.lang = 'pt-BR';
var botaoMais = document.querySelector('.leaflet-control-zoom-in');
var botaoMenos = document.querySelector('.leaflet-control-zoom-out');
if (botaoMais) { botaoMais.title = 'Aproximar'; }
if (botaoMenos) { botaoMenos.title = 'Afastar'; }
'''
mapa_votos.get_root().script.add_child(folium.Element(script_pt))

caminho_mapa_final = os.path.join(base_dir, f"mapa_resultado_{turno_alvo}t_global_2026.html")
mapa_votos.save(caminho_mapa_final)
print(f"Interactive HTML election map exported to: {caminho_mapa_final} ({os.path.getsize(caminho_mapa_final) / 1024 ** 2:,.1f} MB)")

Sections with coordinates and round 2 votes: 499,205
Colors: LULA = #dc3545 | FLAVIO BOLSONARO = #0d6efd
Physical venues on the map: 94,207
GEO_METODO
tse_original             93419
cep_centroid               309
geocodebr_street           153
municipality_centroid      149
nominatim_venue_name        89
nominatim_address           69
geocodebr_locality          19
IBGE 2022 neighborhoods: 17,575 polygons | CRS: EPSG:4674 | columns: ['CD_REGIAO', 'NM_REGIAO', 'CD_UF', 'NM_UF', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST', 'CD_BAIRRO', 'NM_BAIRRO', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI', 'CD_CONCURB', 'NM_CONCURB', 'geometry']

Overseas cities: 144 | boundaries cached: 0 | pending: 144
  ABIDJÃ: boundary suburb (zoom 10) - Le Plateau
  ABU DHABI: boundary city (zoom 10) - المُشرِف
  ACCRA: boundary city (zoom 10) - Acra
  AMSTERDÃ: boundary city (zoom 10) - Amsterdã
  AMÃ: boundary county (zoom 8) - ناحية عمان
  ANCARA: boundary suburb (zoom 10) - Devlet Maha

/mnt/c/Users/daves/OneDrive/Pessoal/Notebooks/tf_env/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'github.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/mnt/c/Users/daves/OneDrive/Pessoal/Notebooks/tf_env/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'release-assets.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
states_2020_simplified.parquet: 100%|██████████| 1.72M/1.72M [00:00<00:00, 6.96MB/s]


Interactive HTML election map exported to: /home/daves/google_drive/Pessoal/Notebooks/Geografia do voto/mapa_resultado_2t_global_2026.html (57.5 MB)
